# Colab Session: opportunity-30m-h24-twosided
Generated from colab-cli history log.

**Session Created**: 2026-10-15 10:00:21
- Endpoint: `m-s-kkb-ass1c0-nte8wkxyawe4`
- Hardware: `NONE`

In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.0/97.1 MB 205.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 78.0/97.1 MB 116.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 114.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 114.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 114.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 114.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 114.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 114.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


In [ ]:
# CELL: CONFIG
from pathlib import Path

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "6h": 360,
}

CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution": "30m",
    "horizon_bars": 24,
    "seed": 42,
    # Bracket. TP must clear the round-trip cost by a wide margin, otherwise no
    # classifier can be profitable (see the notebook header).
    "tp_atr_mult": 4.0,
    "sl_atr_mult": 2.0,
    "atr_period": 14,
    "round_trip_cost_pct": 0.0016,
    # Gate: median TP distance >= ratio x cost. Signal rule: a candle is only
    # eligible when its own TP distance >= ratio x cost (quiet-market guard).
    "min_tp_cost_ratio": 4.0,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    # Decision rule: trade only when P(side) clears a threshold chosen on
    # calibration rows. Candidates are probability quantiles of the calibration set.
    "calib_fraction": 0.15,
    "min_signals": 100,
    "signal_quantiles": [0.99, 0.98, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80],
    "walk_forward_folds": 4,
    # Promotion gate thresholds (held-out test + walk-forward).
    "promote_min_tstat": 2.0,
    "promote_min_wf_positive_share": 0.67,
    "cost_sensitivity_pct": [0.0010, 0.0016, 0.0020],
    # Model capacity. Deliberately regularised: ~90k noisy rows, overlapping labels.
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.03,
    "min_leaf": 200,
    "threads_per_model": 2,
    "ensemble": ["XGB"],
}


def apply_resolution_config(cfg: Dict[str, Any]) -> None:
    """Derive bar minutes, horizon wall-clock, purge, and export_dir from resolution."""
    res = str(cfg["resolution"])
    if res not in RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {res!r}; use one of {sorted(RESOLUTION_MINUTES)}"
        )
    bar_min = int(RESOLUTION_MINUTES[res])
    horizon_bars = int(cfg["horizon_bars"])
    if bar_min <= 0 or horizon_bars <= 0:
        raise ValueError(
            f"horizon_bars={horizon_bars} and {res} bar minutes ({bar_min}) must be positive"
        )
    cfg["bar_minutes"] = bar_min
    cfg["horizon_bars"] = horizon_bars
    cfg["horizon_minutes"] = horizon_bars * bar_min
    # Purge must cover the label look-ahead so no training label peeks into val.
    cfg["purge_bars"] = horizon_bars + 1
    cfg["export_dir"] = f"export/opportunity_{res}_h{horizon_bars}"


_override = Path("opportunity_run_config.json")
if _override.is_file():
    CONFIG.update(json.loads(_override.read_text(encoding="utf-8")))
apply_resolution_config(CONFIG)
CONFIG["ensemble"] = ["XGB"]
CONFIG["export_dir"] = "export/opportunity_30m_h24_twosided"
CONFIG["run_baseline_train"] = False

LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
ENSEMBLE_NAME = "ENSEMBLE"
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution": "30m",
  "horizon_bars": 24,
  "seed": 42,
  "tp_atr_mult": 4.0,
  "sl_atr_mult": 2.0,
  "atr_period": 14,
  "round_trip_cost_pct": 0.0016,
  "min_tp_cost_ratio": 4.0,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calib_fraction": 0.15,
  "min_signals": 100,
  "signal_quantiles": [
    0.99,
    0.98,
    0.97,
    0.95,
    0.92,
    0.9,
    0.85,
    0.8
  ],
  "walk_forward_folds": 4,
  "promote_min_tstat": 2.0,
  "promote_min_wf_positive_share": 0.67,
  "cost_sensitivity_pct": [
    0.001,
    0.0016,
    0.002
  ],
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.03,
  "min_leaf": 200,
  "threads_per_model": 2,
  "ensemble": [
    "XGB"
  ],
  "bar_minutes": 30,
  "horizon_minutes": 720,
  "purge_bars": 25,
  "export_dir": "export/opportunity_30m_h24_twosided",
  "run_baseline_train": false
}


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "6h": 360,
    "1d": 1440,
}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
_resolution = str(CONFIG["resolution"])
# Delta has no native 10m candles; everything else is fetched at its own resolution.
_source_resolution = "5m" if _resolution == "10m" else _resolution
print(
    f"Fetching {CONFIG['symbol']} {_source_resolution} OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']} (target {_resolution})..."
)
raw_bars = fetch_candles(
    str(CONFIG["symbol"]),
    _source_resolution,
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    raw_bars, _source_resolution, symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} {_source_resolution}: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 30m OHLCV for 900d from https://api.india.delta.exchange (target 30m)...


OHLCV BTCUSD 30m: 43200 bars, completeness 100.0%, gaps=0


In [ ]:
# CELL: BARS
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


if str(CONFIG["resolution"]) == "10m":
    bars = build_10m_ohlcv_from_5m(raw_bars)
else:
    bars = raw_bars
if bars is None or getattr(bars, "empty", True):
    raise ValueError(f"No OHLCV bars for {CONFIG['resolution']}")
print(
    f"{CONFIG['resolution']} bars={len(bars)} "
    f"span={bars['time'].iloc[0]} → {bars['time'].iloc[-1]} "
    f"horizon_bars={CONFIG['horizon_bars']} ({CONFIG['horizon_minutes']}m)"
)


30m bars=43200 span=2024-04-28 10:30:00+00:00 → 2026-10-15 10:00:00+00:00 horizon_bars=24 (720m)


In [ ]:
# CELL: FEATURES
OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")


def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(df: pd.DataFrame, *, atr_period: int = 14) -> pd.DataFrame:
    """Causal signal-candle features from the selected timeframe series only.

    Everything is expressed in ATR units or as a ratio so the model is scale
    invariant across the 900-day price range. ATR(14) is also stored for the
    label path. Nothing here looks past the signal candle.
    """
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    atr_safe = atr.replace(0.0, np.nan)
    out["atr"] = atr

    feats: Dict[str, pd.Series] = {}
    # Momentum at several lookbacks, in ATR units.
    feats["ret_1"] = close.pct_change()
    for k in (3, 6, 12, 24, 48, 144):
        feats[f"ret_{k}_atr"] = (close - close.shift(k)) / atr_safe
    # Volatility level and regime.
    feats["atr_pct"] = atr / close.replace(0.0, np.nan)
    atr_long = tr.rolling(int(atr_period) * 6, min_periods=int(atr_period) * 6).mean()
    feats["atr_ratio_84"] = atr / atr_long.replace(0.0, np.nan)
    log_ret = np.log(close).diff()
    rv_12 = log_ret.rolling(12, min_periods=12).std()
    rv_48 = log_ret.rolling(48, min_periods=48).std()
    rv_144 = log_ret.rolling(144, min_periods=144).std()
    feats["rv_12_48"] = rv_12 / rv_48.replace(0.0, np.nan)
    feats["rv_48_144"] = rv_48 / rv_144.replace(0.0, np.nan)
    # Oscillators and trend.
    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)
    for span in (20, 50, 200):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        feats[f"ema_dist_{span}_atr"] = (close - ema) / atr_safe
        feats[f"ema_slope_{span}_atr"] = (ema - ema.shift(6)) / atr_safe
    # Candle anatomy.
    feats["candle_body"] = (close - open_) / atr_safe
    feats["upper_wick"] = (high - np.maximum(close, open_)) / atr_safe
    feats["lower_wick"] = (np.minimum(close, open_) - low) / atr_safe
    feats["range_atr"] = (high - low) / atr_safe
    # Volume anomalies.
    for window in (20, 100):
        vol_mean = volume.rolling(window, min_periods=window).mean()
        vol_std = volume.rolling(window, min_periods=window).std()
        feats[f"volume_z_{window}"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    # Position inside recent ranges and distance to swing extremes.
    for window in (24, 72, 144):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_pos_{window}"] = (close - ll) / (hh - ll).replace(0.0, np.nan)
        feats[f"dist_high_{window}_atr"] = (hh - close) / atr_safe
        feats[f"dist_low_{window}_atr"] = (close - ll) / atr_safe
    # Consecutive same-direction closes (signed).
    sign = np.sign(close.diff()).fillna(0.0)
    groups = (sign != sign.shift()).cumsum()
    feats["streak"] = sign * (sign.groupby(groups).cumcount() + 1)
    # Intraday / weekly seasonality.
    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["dow_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["dow_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    # Short raw history of the last six candle returns (ATR units).
    for lag in range(1, 7):
        feats[f"ret_lag{lag}_atr"] = (close.shift(lag - 1) - close.shift(lag)) / atr_safe
    return pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)


featured = add_opportunity_features(bars, atr_period=int(CONFIG["atr_period"]))
FEATURE_COLS: Tuple[str, ...] = tuple(
    c for c in featured.columns if c not in OHLCV_COLS and c != "atr"
)
print(len(FEATURE_COLS), "features")
null_rates = featured[list(FEATURE_COLS)].isna().mean()
print("max feature null rate", round(float(null_rates.max()), 4), "at", null_rates.idxmax())


46 features
max feature null rate 0.0047 at ema_slope_200_atr


In [ ]:
# CELL: LABELS
# Opportunity labels on a closed-bar series. Path is bars t+1 .. t+horizon only.
# Vectorised over bars; semantics are identical to walking each path bar by bar.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 24
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 4.0
SL_ATR_MULT_DEFAULT = 2.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array where row j-1 holds values shifted back by j bars (NaN padded)."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _side_codes_vectorised(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorised twin of ``tp_sl_from_path`` for every bar at once."""
    denom = np.maximum(atr, _EPS)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry - float(tp_mult) * denom
        sl_level = entry + float(sl_mult) * denom
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_level = entry + float(tp_mult) * denom
        sl_level = entry - float(sl_mult) * denom
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(entry.shape[0])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    codes = np.where(
        ~has_hit,
        NEITHER,
        np.where(tp_first & sl_first, AMBIGUOUS, np.where(tp_first, TP_FIRST, SL_FIRST)),
    )
    return codes.astype(np.int64)


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed bar. Last ``horizon_bars`` rows stay unlabeled.

    Realised P&L columns (``long_pnl_pct`` / ``short_pnl_pct``) are gross of
    cost: +TP distance on TP_FIRST, −SL distance on SL_FIRST **and** AMBIGUOUS
    (a same-bar double touch is scored as a stop), timeout return on NEITHER.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = np.asarray(_causal_atr(out, period=int(atr_period)), dtype=np.float64)
    if atr.shape[0] != n:
        atr = np.resize(atr, n)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close_end = np.full(n, np.nan, dtype=np.float64)
    if n - k > 0:
        fwd_close_end[: n - k] = close[k:]

    index = np.arange(n)
    valid = (
        (index < n - k)
        & np.isfinite(atr)
        & (atr > _EPS)
        & np.isfinite(close)
        & (np.abs(close) > _EPS)
    )
    atr_safe = np.where(valid, atr, np.nan)
    entry_safe = np.where(valid, close, np.nan)

    l_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "LONG")
    s_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "SHORT")
    l_win = l_code == TP_FIRST
    s_win = s_code == TP_FIRST
    label = np.where(l_win & ~s_win, LABEL_LONG, np.where(s_win & ~l_win, LABEL_SHORT, LABEL_NO_TRADE))

    with np.errstate(invalid="ignore", divide="ignore"):
        max_high = np.nanmax(np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0)
        min_low = np.nanmin(np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0)
        upside = np.maximum((max_high - entry_safe) / atr_safe, 0.0)
        downside = np.maximum((entry_safe - min_low) / atr_safe, 0.0)
        timeout_ret = (fwd_close_end - entry_safe) / entry_safe
        tp_pct = float(tp_mult) * atr_safe / entry_safe
        sl_pct = float(sl_mult) * atr_safe / entry_safe
        long_pnl = np.where(
            l_code == TP_FIRST, tp_pct, np.where(l_code == NEITHER, timeout_ret, -sl_pct)
        )
        short_pnl = np.where(
            s_code == TP_FIRST, tp_pct, np.where(s_code == NEITHER, -timeout_ret, -sl_pct)
        )

    def _masked(values: np.ndarray) -> np.ndarray:
        return np.where(valid, values, np.nan)

    out["atr"] = atr
    out["trade_label"] = np.where(valid, label, LABEL_UNLABELED).astype(np.int64)
    out["long_result"] = _masked(l_win.astype(np.float64))
    out["short_result"] = _masked(s_win.astype(np.float64))
    out["long_code"] = _masked(l_code.astype(np.float64))
    out["short_code"] = _masked(s_code.astype(np.float64))
    out["upside"] = _masked(upside)
    out["downside"] = _masked(downside)
    out["timeout_ret"] = _masked(timeout_ret)
    out["tp_pct"] = _masked(tp_pct)
    out["sl_pct"] = _masked(sl_pct)
    out["long_pnl_pct"] = _masked(long_pnl)
    out["short_pnl_pct"] = _masked(short_pnl)
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[out["trade_label"].to_numpy() == LABEL_LONG] = "LONG"
    names[out["trade_label"].to_numpy() == LABEL_SHORT] = "SHORT"
    names[out["trade_label"].to_numpy() == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))
print(
    "median TP distance",
    f"{labeled.loc[valid, 'tp_pct'].median():.4%}",
    "median SL distance",
    f"{labeled.loc[valid, 'sl_pct'].median():.4%}",
    "round trip cost",
    f"{float(CONFIG['round_trip_cost_pct']):.4%}",
)


labeled rows=43163 / 43200
trade_name
NO_TRADE    0.5981
SHORT       0.2027
LONG        0.1992
Name: proportion, dtype: float64
median TP distance 1.6144% median SL distance 0.8072% round trip cost 0.1600%


In [ ]:
# CELL: GATE
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline GATE")
else:
    def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
        cut = int(n * float(train_fraction))
        mask = np.zeros(n, dtype=bool)
        mask[:cut] = True
        return mask


    def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
        finite = usable & np.isfinite(codes)
        return {
            "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
            "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
            "NEITHER": int(np.sum(finite & (codes == NEITHER))),
            "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
        }


    def summarize_label_gate(
        df: pd.DataFrame,
        *,
        train_fraction: float,
        round_trip_cost_pct: float,
        minority_rate: float,
        ambiguous_win_max: float,
        min_tp_cost_ratio: float = 4.0,
    ) -> Dict[str, Any]:
        """Decide whether the bracket is worth training on, using train rows only.

        Besides class mix and oracle exclusive expectancy, the gate checks the
        bracket economics: the median take-profit distance must be at least
        ``min_tp_cost_ratio`` times the round-trip cost, and the implied
        break-even win rate ``(sl + cost) / (tp + sl)`` is reported.
        """
        n = len(df)
        in_train = train_period_mask(n, train_fraction)
        labels = df["trade_label"].to_numpy(dtype=np.int64)
        usable = in_train & (labels >= 0)
        n_use = int(usable.sum())
        if n_use == 0:
            raise RuntimeError("Train-period gate has no labeled rows")
        mix = {}
        mix_ok = True
        for cid, name in enumerate(LABEL_NAMES):
            rate = float(np.mean(labels[usable] == cid))
            mix[name] = rate
            if rate < float(minority_rate):
                mix_ok = False
        long_code = df["long_code"].to_numpy(dtype=np.float64)
        short_code = df["short_code"].to_numpy(dtype=np.float64)
        cost = float(round_trip_cost_pct)
        long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
        short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

        always_long_mask = usable & np.isfinite(long_pnl)
        always_short_mask = usable & np.isfinite(short_pnl)
        always_in_long = (
            float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
        )
        always_in_short = (
            float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
        )

        excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
        excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
        exclusive_long = (
            float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
        )
        exclusive_short = (
            float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
        )

        def _ambig_share(codes: np.ndarray) -> float:
            finite = usable & np.isfinite(codes)
            tp = finite & (codes == TP_FIRST)
            amb = finite & (codes == AMBIGUOUS)
            denom = int(tp.sum() + amb.sum())
            if denom == 0:
                return 0.0
            return float(amb.sum()) / float(denom)

        long_ambig = _ambig_share(long_code)
        short_ambig = _ambig_share(short_code)

        if "tp_pct" in df.columns and "sl_pct" in df.columns:
            tp_pct = df["tp_pct"].to_numpy(dtype=np.float64)
            sl_pct = df["sl_pct"].to_numpy(dtype=np.float64)
        else:
            atr_pct = df["atr"].to_numpy(dtype=np.float64) / df["close"].to_numpy(dtype=np.float64)
            tp_pct = float(CONFIG["tp_atr_mult"]) * atr_pct
            sl_pct = float(CONFIG["sl_atr_mult"]) * atr_pct
        econ_mask = usable & np.isfinite(tp_pct) & np.isfinite(sl_pct)
        median_tp = float(np.median(tp_pct[econ_mask])) if econ_mask.any() else float("nan")
        median_sl = float(np.median(sl_pct[econ_mask])) if econ_mask.any() else float("nan")
        tp_cost_ratio = median_tp / cost if cost > 0 and np.isfinite(median_tp) else float("inf")
        breakeven_win_rate = (
            (median_sl + cost) / (median_tp + median_sl)
            if np.isfinite(median_tp) and np.isfinite(median_sl) and (median_tp + median_sl) > 0
            else float("nan")
        )
        long_tp_rate = float(np.mean(long_code[usable & np.isfinite(long_code)] == TP_FIRST))
        short_tp_rate = float(np.mean(short_code[usable & np.isfinite(short_code)] == TP_FIRST))

        reasons: List[str] = []
        if not mix_ok:
            reasons.append("class_mix_below_floor")
        if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
            reasons.append("exclusive_long_expectancy_not_positive")
        if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
            reasons.append("exclusive_short_expectancy_not_positive")
        if long_ambig >= float(ambiguous_win_max):
            reasons.append("long_ambiguous_majority")
        if short_ambig >= float(ambiguous_win_max):
            reasons.append("short_ambiguous_majority")
        if tp_cost_ratio < float(min_tp_cost_ratio):
            reasons.append("tp_distance_below_cost_ratio")
        passed = len(reasons) == 0
        return {
            "passed": passed,
            "n_train_labeled": n_use,
            "class_mix": mix,
            "minority_rate": float(minority_rate),
            "exclusive_long_expectancy_pct": exclusive_long,
            "exclusive_short_expectancy_pct": exclusive_short,
            "always_in_long_expectancy_pct": always_in_long,
            "always_in_short_expectancy_pct": always_in_short,
            "round_trip_cost_pct": cost,
            "median_tp_pct": median_tp,
            "median_sl_pct": median_sl,
            "tp_cost_ratio": tp_cost_ratio,
            "min_tp_cost_ratio": float(min_tp_cost_ratio),
            "breakeven_win_rate": breakeven_win_rate,
            "base_long_tp_rate": long_tp_rate,
            "base_short_tp_rate": short_tp_rate,
            "long_ambiguous_share": long_ambig,
            "short_ambiguous_share": short_ambig,
            "long_path_codes": _path_code_counts(long_code, usable),
            "short_path_codes": _path_code_counts(short_code, usable),
            "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
            "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
            "reasons": reasons,
        }


    EXPORT_DIR = Path(str(CONFIG["export_dir"]))
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    label_report = summarize_label_gate(
        labeled,
        train_fraction=float(CONFIG["train_fraction"]),
        round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
        minority_rate=float(CONFIG["minority_rate"]),
        ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
        min_tp_cost_ratio=float(CONFIG["min_tp_cost_ratio"]),
    )
    report_path = EXPORT_DIR / "label_report.json"
    report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
    print(json.dumps(label_report, indent=2))
    print("wrote", report_path)
    if not label_report["passed"]:
        raise RuntimeError(
            "Train-period label gate failed: "
            f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples or horizon and "
            "re-run from the top. Do not fit models."
        )
    print(
        f"Gate passed. Break-even win rate {label_report['breakeven_win_rate']:.1%} vs base "
        f"TP rates long {label_report['base_long_tp_rate']:.1%} / short "
        f"{label_report['base_short_tp_rate']:.1%}. Training may start."
    )


Skipping baseline GATE


In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


def inner_calibration_split(
    train_idx: np.ndarray, *, calib_fraction: float, purge_bars: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Reserve the purged tail of a training window for threshold selection."""
    n = len(train_idx)
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = train_idx[: max(cut - int(purge_bars), 0)]
    calib_idx = train_idx[cut:]
    return fit_idx, calib_idx


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=int(CONFIG["walk_forward_folds"]), purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 30214 range 0 30213
val n 6455 range 30264 36718
test n 6431 range 36769 43199
walk-forward folds 3 dev_end 36769


In [ ]:
# CELL: DATASET
def build_rows(
    frame: pd.DataFrame, *, feature_cols: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One tabular row per labeled bar with finite features and finite path targets."""
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    ok = (y_cls >= 0) & np.isfinite(feats).all(axis=1) & np.isfinite(y_reg).all(axis=1)
    rows = np.flatnonzero(ok).astype(np.int64)
    return (
        feats[rows].astype(np.float32),
        y_cls[rows],
        y_reg[rows].astype(np.float32),
        rows,
    )


all_x, all_y, all_r, all_rows = build_rows(labeled, feature_cols=FEATURE_COLS)
print(
    "rows",
    all_x.shape,
    "class mix",
    {n: round(float(np.mean(all_y == i)), 4) for i, n in enumerate(LABEL_NAMES)},
)


def rows_in(split_idx: np.ndarray) -> np.ndarray:
    """Boolean selector over ``all_rows`` for a set of frame indices."""
    allowed = np.zeros(len(labeled), dtype=bool)
    allowed[np.asarray(split_idx, dtype=np.int64)] = True
    return allowed[all_rows]


MIN_SIGNAL_TP_PCT = float(CONFIG["min_tp_cost_ratio"]) * float(CONFIG["round_trip_cost_pct"])


def signal_eligible(rows: np.ndarray) -> np.ndarray:
    """Quiet-market guard: a candle may carry a signal only if its TP distance pays for cost."""
    tp_pct = labeled["tp_pct"].to_numpy(dtype=np.float64)[rows]
    return np.isfinite(tp_pct) & (tp_pct >= MIN_SIGNAL_TP_PCT)


print(
    f"signal eligibility: TP distance >= {MIN_SIGNAL_TP_PCT:.4%} -> "
    f"{float(signal_eligible(all_rows).mean()):.1%} of labeled candles"
)


def fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


sel_train = rows_in(splits["train"])
sel_val = rows_in(splits["val"])
sel_test = rows_in(splits["test"])
feat_mean, feat_std = fit_scaler(all_x[sel_train])
reg_mean = all_r[sel_train].mean(axis=0, keepdims=True)
reg_std = all_r[sel_train].std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


X_train = scale_x(all_x[sel_train], feat_mean, feat_std)
X_val = scale_x(all_x[sel_val], feat_mean, feat_std)
X_test = scale_x(all_x[sel_test], feat_mean, feat_std)
y_train, y_val, y_test = all_y[sel_train], all_y[sel_val], all_y[sel_test]
r_train, r_val, r_test = all_r[sel_train], all_r[sel_val], all_r[sel_test]
R_train, R_val, R_test = scale_r(r_train), scale_r(r_val), scale_r(r_test)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)


rows (42971, 46) class mix {'LONG': 0.1991, 'SHORT': 0.202, 'NO_TRADE': 0.5989}
signal eligibility: TP distance >= 0.6400% -> 95.3% of labeled candles
train (30009, 46) val (6455, 46) test (6407, 46)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family.

    No class re-weighting anywhere: the decision rule thresholds calibrated-ish
    probabilities, and ``balanced`` weights distort them.
    """
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    threads = int(CONFIG["threads_per_model"])
    if name == "LR":
        clf = LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")
        up = Ridge(alpha=10.0)
        dn = Ridge(alpha=10.0)
    elif name == "RF":
        rf_kwargs = dict(
            n_estimators=est,
            max_depth=depth + 2,
            min_samples_leaf=leaf,
            max_features="sqrt",
            n_jobs=threads,
        )
        clf = RandomForestClassifier(random_state=seed, **rf_kwargs)
        up = RandomForestRegressor(random_state=seed, **rf_kwargs)
        dn = RandomForestRegressor(random_state=seed + 1, **rf_kwargs)
    elif name == "XGB":
        xgb_kwargs = dict(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_weight=leaf,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            tree_method="hist",
            n_jobs=threads,
            verbosity=0,
        )
        clf = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", random_state=seed, **xgb_kwargs
        )
        up = XGBRegressor(random_state=seed, **xgb_kwargs)
        dn = XGBRegressor(random_state=seed + 1, **xgb_kwargs)
    elif name == "LGBM":
        lgb_kwargs = dict(
            n_estimators=est,
            num_leaves=31,
            max_depth=-1,
            learning_rate=lr,
            min_child_samples=leaf,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            n_jobs=threads,
            verbosity=-1,
        )
        clf = LGBMClassifier(random_state=seed, **lgb_kwargs)
        up = LGBMRegressor(random_state=seed, **lgb_kwargs)
        dn = LGBMRegressor(random_state=seed + 1, **lgb_kwargs)
    elif name == "CatBoost":
        cb_kwargs = dict(
            iterations=est,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=10.0,
            verbose=False,
            thread_count=threads,
            allow_writing_files=False,
        )
        clf = CatBoostClassifier(loss_function="MultiClass", random_seed=seed, **cb_kwargs)
        up = CatBoostRegressor(random_seed=seed, **cb_kwargs)
        dn = CatBoostRegressor(random_seed=seed + 1, **cb_kwargs)
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"], "+", ENSEMBLE_NAME)


families ['XGB'] + ENSEMBLE


In [ ]:
# CELL: TRAIN
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline TRAIN")
else:
    def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
        mat = np.zeros((n_classes, n_classes), dtype=np.int64)
        for t, p in zip(y_true.tolist(), y_pred.tolist()):
            if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
                mat[int(t), int(p)] += 1
        return mat


    def class_precision(mat: np.ndarray, cls: int) -> float:
        col = mat[:, cls]
        denom = float(col.sum())
        if denom <= 0:
            return float("nan")
        return float(mat[cls, cls]) / denom


    def balanced_accuracy(mat: np.ndarray) -> float:
        recalls = []
        for i in range(mat.shape[0]):
            denom = float(mat[i].sum())
            recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
        finite = [r for r in recalls if np.isfinite(r)]
        return float(np.mean(finite)) if finite else float("nan")


    def corr(a: np.ndarray, b: np.ndarray) -> float:
        if a.size < 3:
            return float("nan")
        if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
            return float("nan")
        return float(np.corrcoef(a, b)[0, 1])


    def as_cls(arr: np.ndarray) -> np.ndarray:
        return np.asarray(arr).reshape(-1).astype(np.int64)


    def as_1d(arr: np.ndarray) -> np.ndarray:
        return np.asarray(arr, dtype=np.float64).reshape(-1)


    def proba_of(clf: Any, X: np.ndarray) -> np.ndarray:
        """(n, 3) class probabilities in LONG/SHORT/NO_TRADE order regardless of estimator."""
        p = np.asarray(clf.predict_proba(X), dtype=np.float64)
        if p.ndim == 1:
            p = p.reshape(len(X), -1)
        classes = getattr(clf, "classes_", None)
        if classes is None:
            return p
        full = np.zeros((p.shape[0], len(LABEL_NAMES)), dtype=np.float64)
        for j, c in enumerate(np.asarray(classes).reshape(-1).astype(int)):
            full[:, c] = p[:, j]
        return full


    def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
        cm = confusion(y_true, y_pred)
        return {
            "precision_LONG": class_precision(cm, LABEL_LONG),
            "precision_SHORT": class_precision(cm, LABEL_SHORT),
            "n_LONG": float(np.sum(y_pred == LABEL_LONG)),
            "n_SHORT": float(np.sum(y_pred == LABEL_SHORT)),
            "balanced_accuracy": balanced_accuracy(cm),
            "confusion": cm.tolist(),
        }


    _PNL_COL = {LABEL_LONG: "long_pnl_pct", LABEL_SHORT: "short_pnl_pct"}
    _SIDE_NAME = {LABEL_LONG: "LONG", LABEL_SHORT: "SHORT"}


    def net_pnl(cls: int, rows: np.ndarray, cost: Optional[float] = None) -> np.ndarray:
        """Net-of-cost realised P&L (fraction of price) for taking ``cls`` on ``rows``."""
        c = float(CONFIG["round_trip_cost_pct"]) if cost is None else float(cost)
        return labeled[_PNL_COL[int(cls)]].to_numpy(dtype=np.float64)[rows] - c


    def signal_expectancy(
        pred_cls: np.ndarray, rows: np.ndarray, cost: Optional[float] = None
    ) -> Dict[str, float]:
        """Per-side expectancy, t-stat, profit factor and total for the predicted signals."""
        out: Dict[str, float] = {}
        for cls, name in _SIDE_NAME.items():
            take = pred_cls == cls
            pnl = net_pnl(cls, rows[take], cost) if take.any() else np.zeros(0)
            pnl = pnl[np.isfinite(pnl)]
            n = int(pnl.size)
            out[f"{name}_n"] = float(n)
            out[f"{name}_expectancy"] = float(pnl.mean()) if n else float("nan")
            std = float(pnl.std(ddof=1)) if n > 1 else float("nan")
            out[f"{name}_tstat"] = (
                float(pnl.mean() / (std / math.sqrt(n))) if n > 1 and std > 0 else float("nan")
            )
            wins = float(pnl[pnl > 0].sum())
            losses = float(-pnl[pnl < 0].sum())
            out[f"{name}_profit_factor"] = (
                wins / losses if losses > 0 else (float("inf") if wins > 0 else float("nan"))
            )
            out[f"{name}_total_pct"] = float(pnl.sum())
        return out


    def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
        """Mean expectancy over sides with >= ``min_signals`` trades; -inf when none qualify."""
        min_n = int(CONFIG["min_signals"])
        parts: List[float] = []
        for name in ("LONG", "SHORT"):
            n = float(exp_report.get(f"{name}_n", 0.0))
            e = float(exp_report.get(f"{name}_expectancy", float("nan")))
            if n >= min_n and np.isfinite(e):
                parts.append(e)
        return float(np.mean(parts)) if parts else float("-inf")


    def choose_thresholds(proba: np.ndarray, rows: np.ndarray) -> Dict[str, Optional[float]]:
        """Per side, the probability threshold that maximises calibration expectancy.

        Candidates are quantiles of the side's probability on the calibration rows.
        A side with no candidate reaching ``min_signals`` trades is disabled (None).
        """
        min_n = int(CONFIG["min_signals"])
        eligible = signal_eligible(rows)
        out: Dict[str, Optional[float]] = {}
        for cls, name in _SIDE_NAME.items():
            other = LABEL_SHORT if cls == LABEL_LONG else LABEL_LONG
            p_side = proba[:, cls]
            p_other = proba[:, other]
            pnl_all = net_pnl(cls, rows)
            best_thr: Optional[float] = None
            best_e = float("-inf")
            if not eligible.any():
                out[name] = None
                continue
            for q in CONFIG["signal_quantiles"]:
                thr = float(np.quantile(p_side[eligible], float(q)))
                take = eligible & (p_side >= thr) & (p_side > p_other) & np.isfinite(pnl_all)
                n = int(take.sum())
                if n < min_n:
                    continue
                e = float(pnl_all[take].mean())
                if e > best_e:
                    best_thr, best_e = thr, e
            out[name] = best_thr
        return out


    def apply_thresholds(
        proba: np.ndarray, thr: Dict[str, Optional[float]], eligible: Optional[np.ndarray] = None
    ) -> np.ndarray:
        """Live decision rule.

        Abstain unless the candle is eligible (TP distance pays for cost), P(side)
        clears the side's threshold and beats the other side.
        """
        pred = np.full(proba.shape[0], LABEL_NO_TRADE, dtype=np.int64)
        ok = np.ones(proba.shape[0], dtype=bool) if eligible is None else np.asarray(eligible, dtype=bool)
        p_long, p_short = proba[:, LABEL_LONG], proba[:, LABEL_SHORT]
        if thr.get("LONG") is not None:
            pred[ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
        if thr.get("SHORT") is not None:
            pred[ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
        return pred


    def score_probas(
        p_calib: np.ndarray,
        rows_calib: np.ndarray,
        p_eval: np.ndarray,
        rows_eval: np.ndarray,
        y_eval: np.ndarray,
    ) -> Dict[str, Any]:
        thr = choose_thresholds(p_calib, rows_calib)
        pred = apply_thresholds(p_eval, thr, signal_eligible(rows_eval))
        exp_rep = signal_expectancy(pred, rows_eval)
        cls_rep = classifier_report(y_eval, pred)
        return {
            "thresholds": thr,
            "cls": {k: v for k, v in cls_rep.items() if k != "confusion"},
            "confusion": cls_rep["confusion"],
            "exp": exp_rep,
            "score": expectancy_rank_score(exp_rep),
            "pred": pred,
        }


    def fit_fold(
        name: str, fold: str, fit_sel: np.ndarray, calib_sel: np.ndarray, eval_sel: np.ndarray, *, seed: int
    ) -> Dict[str, Any]:
        """Fit one family on ``fit_sel`` rows, return probabilities on calibration and eval rows."""
        mean, std = fit_scaler(all_x[fit_sel])
        pack = make_family(name, seed)
        pack["clf"].fit(scale_x(all_x[fit_sel], mean, std), all_y[fit_sel])
        result = {
            "name": name,
            "fold": fold,
            "clf": pack["clf"],
            "scaler": (mean, std),
            "p_calib": proba_of(pack["clf"], scale_x(all_x[calib_sel], mean, std)),
            "p_eval": proba_of(pack["clf"], scale_x(all_x[eval_sel], mean, std)),
        }
        if fold == "final":
            pack["up"].fit(X_train, R_train[:, 0])
            pack["dn"].fit(X_train, R_train[:, 1])
            result["up"] = pack["up"]
            result["dn"] = pack["dn"]
        return result


    # Fold plan: expanding walk-forward folds inside development, then the final split.
    # Every fold has disjoint fit / calibration / evaluation rows; thresholds are chosen
    # on calibration rows only, so evaluation expectancy is out-of-sample for both the
    # model and the decision rule.
    fold_plan: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray]] = []
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_idx, calib_idx = inner_calibration_split(
            tr_idx, calib_fraction=float(CONFIG["calib_fraction"]), purge_bars=int(CONFIG["purge_bars"])
        )
        fold_plan.append((f"wf{fold_i}", rows_in(fit_idx), rows_in(calib_idx), rows_in(va_idx)))
    fold_plan.append(("final", sel_train, sel_val, sel_test))
    for fold, f_sel, c_sel, e_sel in fold_plan:
        print(f"{fold}: fit={int(f_sel.sum())} calib={int(c_sel.sum())} eval={int(e_sel.sum())}")

    FAMILIES: List[str] = list(CONFIG["ensemble"])
    runs: Dict[str, Dict[str, Dict[str, Any]]] = {fold: {} for fold, *_ in fold_plan}
    print("Fitting", FAMILIES, "x", [f for f, *_ in fold_plan], "in parallel...")
    t_start = time.time()
    with ThreadPoolExecutor(max_workers=len(FAMILIES)) as pool:
        futs = {}
        for fold_i, (fold, f_sel, c_sel, e_sel) in enumerate(fold_plan):
            for name in FAMILIES:
                fut = pool.submit(
                    fit_fold, name, fold, f_sel, c_sel, e_sel, seed=int(CONFIG["seed"]) + fold_i
                )
                futs[fut] = (fold, name)
        for fut in as_completed(futs):
            fold, name = futs[fut]
            try:
                runs[fold][name] = fut.result()
                print(f"  done {name:9s} {fold}  ({time.time() - t_start:.0f}s)")
            except Exception as exc:  # keep the bake-off alive if one family fails
                print(f"  FAILED {name} {fold}: {exc!r}")

    # Soft-vote ensemble over the members that succeeded in a fold.
    for fold, f_sel, c_sel, e_sel in fold_plan:
        members = [runs[fold][n] for n in FAMILIES if n in runs[fold]]
        if len(members) >= 2:
            runs[fold][ENSEMBLE_NAME] = {
                "name": ENSEMBLE_NAME,
                "fold": fold,
                "members": [m["name"] for m in members],
                "p_calib": np.mean([m["p_calib"] for m in members], axis=0),
                "p_eval": np.mean([m["p_eval"] for m in members], axis=0),
            }

    CANDIDATES: List[str] = [n for n in FAMILIES + [ENSEMBLE_NAME] if n in runs["final"]]
    scores: Dict[str, Dict[str, Dict[str, Any]]] = {name: {} for name in CANDIDATES}
    for fold, f_sel, c_sel, e_sel in fold_plan:
        rows_c, rows_e, y_e = all_rows[c_sel], all_rows[e_sel], all_y[e_sel]
        for name in CANDIDATES:
            if name not in runs[fold]:
                continue
            r = runs[fold][name]
            scores[name][fold] = score_probas(r["p_calib"], rows_c, r["p_eval"], rows_e, y_e)

    WF_FOLDS: List[str] = [fold for fold, *_ in fold_plan if fold != "final"]


    def walk_forward_summary(name: str) -> Dict[str, float]:
        vals = [scores[name][f]["score"] for f in WF_FOLDS if f in scores[name]]
        finite = [v for v in vals if np.isfinite(v)]
        return {
            "wf_folds_scored": float(len(finite)),
            "wf_mean_score": float(np.mean(finite)) if finite else float("-inf"),
            "wf_positive_share": float(np.mean([v > 0 for v in finite])) if finite else 0.0,
            "wf_min_score": float(np.min(finite)) if finite else float("-inf"),
        }


    leaderboard: Dict[str, Dict[str, Any]] = {}
    for name in CANDIDATES:
        wf = walk_forward_summary(name)
        fin = scores[name].get("final")
        row: Dict[str, Any] = {
            **wf,
            "val_thresholds": fin["thresholds"] if fin else None,
            "wf_folds": {
                f: {"score": scores[name][f]["score"], "exp": scores[name][f]["exp"], "thresholds": scores[name][f]["thresholds"]}
                for f in WF_FOLDS
                if f in scores[name]
            },
        }
        if name != ENSEMBLE_NAME and "up" in runs["final"].get(name, {}):
            r = runs["final"][name]
            up_c = corr(as_1d(r["up"].predict(X_val)), R_val[:, 0])
            dn_c = corr(as_1d(r["dn"].predict(X_val)), R_val[:, 1])
            row["upside_corr"] = up_c
            row["downside_corr"] = dn_c
            row["reg_score"] = float(np.nanmean([up_c, dn_c]))
        leaderboard[name] = row
        print(
            f"{name:9s} WF mean={wf['wf_mean_score']:+.6f} min={wf['wf_min_score']:+.6f} "
            f"pos_share={wf['wf_positive_share']:.2f} folds={wf['wf_folds_scored']:.0f} "
            f"val_thr={row['val_thresholds']} reg={row.get('reg_score', float('nan')):.3f}"
        )


    def _selection_key(name: str) -> Tuple[float, float]:
        wf = leaderboard[name]
        return (wf["wf_mean_score"], wf["wf_min_score"])


    cls_winner_name = max(CANDIDATES, key=_selection_key)
    reg_candidates = [n for n in CANDIDATES if "reg_score" in leaderboard[n]]
    reg_winner_name = max(reg_candidates, key=lambda n: float(leaderboard[n]["reg_score"]))
    cls_winner = {"name": cls_winner_name, **leaderboard[cls_winner_name]}
    reg_winner = {"name": reg_winner_name, "reg_score": leaderboard[reg_winner_name]["reg_score"]}
    print("best classifier (walk-forward):", cls_winner_name, "| best regressor family (val):", reg_winner_name)


Skipping baseline TRAIN


In [ ]:
# CELL: EVAL
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline EVAL")
else:
    def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
        return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


    rows_test = all_rows[sel_test]
    final_scores = {name: scores[name]["final"] for name in CANDIDATES}
    win = final_scores[cls_winner_name]
    pred_cls = win["pred"]
    cm = np.asarray(win["confusion"])
    print("TEST winner clf=", cls_winner_name, "thresholds=", win["thresholds"], "reg=", reg_winner_name)
    print(
        f"test candles eligible for a signal (TP distance >= {MIN_SIGNAL_TP_PCT:.4%}): "
        f"{float(signal_eligible(rows_test).mean()):.1%}"
    )
    print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
    print(cm)
    print("test signals", {k: v for k, v in win["exp"].items()})

    # Cost sensitivity of the winner's test signals (same decisions, different cost).
    cost_sensitivity = {}
    for c in CONFIG["cost_sensitivity_pct"]:
        rep = signal_expectancy(pred_cls, rows_test, cost=float(c))
        cost_sensitivity[f"{float(c):.4f}"] = {
            "score": expectancy_rank_score(rep),
            "LONG_expectancy": rep["LONG_expectancy"],
            "SHORT_expectancy": rep["SHORT_expectancy"],
        }
        print(f"cost {float(c):.4%}: score={cost_sensitivity[f'{float(c):.4f}']['score']:+.6f}")

    member_test: Dict[str, Dict[str, Any]] = {}
    for name in CANDIDATES:
        s = final_scores[name]
        member_test[name] = {**s["cls"], "exp": s["exp"], "score": s["score"], "thresholds": s["thresholds"]}
        if name != ENSEMBLE_NAME and "up" in runs["final"][name]:
            r = runs["final"][name]
            member_test[name]["upside_corr"] = corr(_unscale_reg(r["up"].predict(X_test), 0), r_test[:, 0])
            member_test[name]["downside_corr"] = corr(_unscale_reg(r["dn"].predict(X_test), 1), r_test[:, 1])
        print(
            name,
            "test",
            {
                "score": round(s["score"], 6) if np.isfinite(s["score"]) else s["score"],
                "LONG": (int(s["exp"]["LONG_n"]), round(s["exp"]["LONG_expectancy"], 6) if np.isfinite(s["exp"]["LONG_expectancy"]) else None),
                "SHORT": (int(s["exp"]["SHORT_n"]), round(s["exp"]["SHORT_expectancy"], 6) if np.isfinite(s["exp"]["SHORT_expectancy"]) else None),
                "P_LONG": round(s["cls"]["precision_LONG"], 3) if np.isfinite(s["cls"]["precision_LONG"]) else None,
                "P_SHORT": round(s["cls"]["precision_SHORT"], 3) if np.isfinite(s["cls"]["precision_SHORT"]) else None,
            },
        )

    best_up = runs["final"][reg_winner_name]["up"]
    best_dn = runs["final"][reg_winner_name]["dn"]
    up_hat = _unscale_reg(best_up.predict(X_test), 0)
    dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
    naive_up_mae = float(np.mean(np.abs(r_test[:, 0] - float(reg_mean[0, 0]))))
    naive_dn_mae = float(np.mean(np.abs(r_test[:, 1] - float(reg_mean[0, 1]))))
    reg_eval = {
        "upside_corr": corr(up_hat, r_test[:, 0]),
        "downside_corr": corr(dn_hat, r_test[:, 1]),
        "upside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 0] - up_hat))) / naive_up_mae,
        "downside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 1] - dn_hat))) / naive_dn_mae,
    }
    print("regression test", reg_eval)

    eval_report = {
        "n_test": int(len(y_test)),
        "best_classifier": cls_winner_name,
        "best_regressor_family": reg_winner_name,
        "decision_rule": {
            "type": "probability_threshold_abstain",
            "thresholds": win["thresholds"],
            "min_signals": int(CONFIG["min_signals"]),
            "min_signal_tp_pct": MIN_SIGNAL_TP_PCT,
            "test_eligible_share": float(signal_eligible(rows_test).mean()),
        },
        "val_leaderboard": leaderboard,
        "test_members": member_test,
        "confusion": cm.tolist(),
        "precision_LONG": win["cls"]["precision_LONG"],
        "precision_SHORT": win["cls"]["precision_SHORT"],
        "n_LONG": win["cls"]["n_LONG"],
        "n_SHORT": win["cls"]["n_SHORT"],
        "balanced_accuracy": win["cls"]["balanced_accuracy"],
        "signal_expectancy": win["exp"],
        "score": win["score"],
        "cost_sensitivity": cost_sensitivity,
        "regression": reg_eval,
        "note": (
            "Classifier chosen by mean walk-forward thresholded expectancy (both sides, "
            f"min {int(CONFIG['min_signals'])} signals per side). Thresholds for the test come from "
            "validation only. Test is scored once. Balanced accuracy is diagnostic only."
        ),
    }
    print(json.dumps({k: v for k, v in eval_report.items() if k not in ("val_leaderboard", "test_members")}, indent=2, default=float))


Skipping baseline EVAL


In [ ]:
# CELL: PROMOTE
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline PROMOTE")
else:
    def promotion_gate(eval_rep: Dict[str, Any], lb_row: Dict[str, Any]) -> Dict[str, Any]:
        """Decide ``ready_to_promote`` from held-out evidence only.

        Requirements: positive thresholded expectancy on the single held-out test,
        every active side has >= ``min_signals`` trades and t-stat >= ``promote_min_tstat``,
        expectancy stays positive at the highest cost in ``cost_sensitivity_pct``, and the
        walk-forward positive share is >= ``promote_min_wf_positive_share``.
        """
        reasons: List[str] = []
        min_n = int(CONFIG["min_signals"])
        min_t = float(CONFIG["promote_min_tstat"])
        exp = eval_rep["signal_expectancy"]
        active_sides = [s for s in ("LONG", "SHORT") if float(exp.get(f"{s}_n", 0.0)) > 0]
        if not active_sides:
            reasons.append("no_signals_on_test")
        if not (np.isfinite(eval_rep["score"]) and eval_rep["score"] > 0.0):
            reasons.append("test_expectancy_not_positive")
        for side in active_sides:
            if float(exp[f"{side}_n"]) < min_n:
                reasons.append(f"{side.lower()}_too_few_test_signals")
            t = float(exp.get(f"{side}_tstat", float("nan")))
            if not (np.isfinite(t) and t >= min_t):
                reasons.append(f"{side.lower()}_tstat_below_{min_t:g}")
            if not (np.isfinite(exp[f"{side}_expectancy"]) and exp[f"{side}_expectancy"] > 0.0):
                reasons.append(f"{side.lower()}_expectancy_not_positive")
        worst_cost = max(float(c) for c in CONFIG["cost_sensitivity_pct"])
        worst = eval_rep["cost_sensitivity"][f"{worst_cost:.4f}"]["score"]
        if not (np.isfinite(worst) and worst > 0.0):
            reasons.append("fragile_to_higher_cost")
        if float(lb_row.get("wf_positive_share", 0.0)) < float(CONFIG["promote_min_wf_positive_share"]):
            reasons.append("walk_forward_not_consistently_positive")
        if float(lb_row.get("wf_folds_scored", 0.0)) < 2:
            reasons.append("walk_forward_insufficient_folds")
        return {
            "ready_to_promote": len(reasons) == 0,
            "reasons": reasons,
            "test_score": eval_rep["score"],
            "test_signals": {s: exp.get(f"{s}_n") for s in ("LONG", "SHORT")},
            "test_tstats": {s: exp.get(f"{s}_tstat") for s in ("LONG", "SHORT")},
            "wf_positive_share": lb_row.get("wf_positive_share"),
            "wf_mean_score": lb_row.get("wf_mean_score"),
            "worst_cost_score": worst,
        }


    promotion = promotion_gate(eval_report, leaderboard[cls_winner_name])
    ready_to_promote = bool(promotion["ready_to_promote"])
    print(json.dumps(promotion, indent=2, default=float))
    if ready_to_promote:
        print("VERDICT: held-out evidence supports paper-trading this bundle (still research-only here).")
    else:
        print("VERDICT: NOT ready. Do not trade this bundle live. Reasons:", promotion["reasons"])


Skipping baseline PROMOTE


In [ ]:
# CELL: EXPORT
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline EXPORT")
else:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    final_run = runs["final"][cls_winner_name]
    if cls_winner_name == ENSEMBLE_NAME:
        clf_payload: Any = {
            "members": {m: runs["final"][m]["clf"] for m in final_run["members"]},
            "member_scalers": {m: runs["final"][m]["scaler"] for m in final_run["members"]},
        }
    else:
        clf_payload = {"members": {cls_winner_name: final_run["clf"]}, "member_scalers": {cls_winner_name: final_run["scaler"]}}
    bundle = {
        "best_classifier": cls_winner_name,
        "best_regressor_family": reg_winner_name,
        "clf": clf_payload,
        "thresholds": eval_report["decision_rule"]["thresholds"],
        "up": best_up,
        "dn": best_dn,
        "feat_mean": feat_mean,
        "feat_std": feat_std,
        "reg_mean": reg_mean,
        "reg_std": reg_std,
        "feature_cols": list(FEATURE_COLS),
        "label_names": list(LABEL_NAMES),
    }
    with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
        pickle.dump(bundle, handle)
    (EXPORT_DIR / "leaderboard.json").write_text(
        json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
    )
    metadata = {
        "model_family": f"jacksparrow_opportunity_{CONFIG['resolution']}_tabular_ensemble",
        "ensemble": list(CONFIG["ensemble"]),
        "candidates": CANDIDATES,
        "best_classifier": cls_winner_name,
        "best_regressor_family": reg_winner_name,
        "timeframe": str(CONFIG["resolution"]),
        "resolution": str(CONFIG["resolution"]),
        "horizon_bars": int(CONFIG["horizon_bars"]),
        "horizon_minutes": int(CONFIG["horizon_minutes"]),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
        "feature_cols": list(FEATURE_COLS),
        "label_names": list(LABEL_NAMES),
        "decision_rule": {
            **eval_report["decision_rule"],
            "description": (
                f"At a closed {CONFIG['resolution']} candle: if tp_atr_mult * ATR(14) / close < min_signal_tp_pct emit "
                "NO_TRADE (quiet-market guard). Otherwise scale features with the member scaler(s), "
                "average member predict_proba when best_classifier == ENSEMBLE, then emit LONG when "
                "P(LONG) >= thresholds.LONG and P(LONG) > P(SHORT); SHORT symmetrically; else NO_TRADE. "
                "A null threshold disables that side."
            ),
        },
        "ready_to_promote": ready_to_promote,
        "promote_reason": (
            "held_out_gate_passed_paper_trade_first" if ready_to_promote else ";".join(promotion["reasons"])
        ),
        "promotion": promotion,
        "eval": eval_report,
        "label_gate": label_report,
        "walk_forward": {name: leaderboard[name]["wf_folds"] for name in CANDIDATES},
        "config": CONFIG,
    }
    (EXPORT_DIR / "metadata.json").write_text(
        json.dumps(metadata, indent=2, default=float), encoding="utf-8"
    )
    (EXPORT_DIR / "eval_report.json").write_text(
        json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
    )
    print("wrote", EXPORT_DIR / "best_models.pkl")
    print("best classifier", cls_winner_name, "best regressor", reg_winner_name)
    print("ready_to_promote=", ready_to_promote)
    try:
        from google.colab import files
        import shutil

        shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
        files.download(str(EXPORT_DIR) + ".zip")
    except ImportError:
        print("Not Colab; files remain under", EXPORT_DIR)



Skipping baseline EXPORT


In [ ]:
# CELL: FL_SPECS
"""Frozen 50-feature 30m research: two-sided 4/2 ATR policy and MLP.

Features stay fixed. Long and short opportunity, volatility, and excursion
are trained into a long/short/flat policy. Injected as ``# CELL: FL_SPECS``.
"""

from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")
FROZEN_HORIZON_BARS = 24
FROZEN_RESOLUTION = "30m"
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT = 4.0
SL_ATR_MULT = 2.0
DIR_BAND_ATR_MULT = 0.5
SCREEN_SEED = 42
N_ESTIMATORS_DEFAULT = 400
_EPS = 1e-12
R_EPS = 1e-12
UNLABELED = -1
MIN_SIGNALS_DEFAULT = 100
OPP_QUANTILES: Tuple[float, ...] = (0.5, 0.8, 0.9, 0.95)
K_VOL_GRID: Tuple[float, ...] = (0.15, 0.25, 0.35)
K_DD_GRID: Tuple[float, ...] = (0.8, 1.0, 1.2)
MLP_HIDDEN: Tuple[int, int] = (64, 32)
MLP_EPOCHS = 25
MLP_BATCH = 256
MLP_LR = 1e-3

DIR_UP = 0
DIR_NEUTRAL = 1
DIR_DOWN = 2
DIR_NAMES: Tuple[str, ...] = ("UP", "NEUTRAL", "DOWN")

OPP_TP_FIRST = 0
OPP_SL_FIRST = 1
OPP_TIMEOUT = 2
OPP_NAMES: Tuple[str, ...] = ("TP_FIRST", "SL_FIRST", "TIMEOUT")

FIXED_FEATURE_COLS: Tuple[str, ...] = (
    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_12",
    "return_24",
    "return_48",
    "candle_body",
    "range_pct",
    "upper_wick_pct",
    "lower_wick_pct",
    "close_location",
    "ema_distance_8",
    "ema_distance_16",
    "ema_distance_24",
    "ema_distance_48",
    "ema_slope_8",
    "ema_slope_24",
    "rsi_7",
    "rsi_14",
    "rsi_28",
    "macd",
    "macd_signal_distance",
    "adx_14",
    "atr_pct_14",
    "rv_4",
    "rv_12",
    "rv_24",
    "volatility_ratio",
    "bollinger_z_20",
    "range_position_24",
    "range_position_48",
    "breakout_distance_24",
    "log_volume",
    "volume_change",
    "volume_zscore_24",
    "signed_volume_proxy",
    "volume_trend_24",
    "consecutive_candle_direction",
    "higher_high_count_8",
    "lower_low_count_8",
    "trend_efficiency_24",
    "rolling_return_skew_24",
    "rolling_return_kurtosis_24",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "drawdown_48",
    "distance_from_rolling_extreme",
)

TARGET_SPECS: Tuple[Dict[str, Any], ...] = (
    {
        "id": "direction_head",
        "kind": "classification",
        "y_col": "y_direction",
        "n_classes": 3,
        "names": DIR_NAMES,
    },
    {
        "id": "return_head",
        "kind": "regression",
        "y_col": "y_return",
    },
    {
        "id": "volatility_head",
        "kind": "regression",
        "y_col": "y_volatility",
    },
    {
        "id": "opportunity_head",
        "kind": "classification",
        "y_col": "y_opportunity",
        "n_classes": 3,
        "names": OPP_NAMES,
    },
    {
        "id": "opportunity_short_head",
        "kind": "classification",
        "y_col": "y_opportunity_short",
        "n_classes": 3,
        "names": OPP_NAMES,
    },
    {
        "id": "excursion_head",
        "kind": "regression_pair",
        "y_cols": ("y_upside", "y_downside"),
    },
)

POLICY_HEAD_IDS: Tuple[str, ...] = (
    "opportunity_head",
    "opportunity_short_head",
    "volatility_head",
    "excursion_head",
)


def _require_ohlcv(df: pd.DataFrame) -> None:
    missing = [c for c in OHLCV_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"OHLCV frame missing columns: {missing}")


def _wilder_rsi(close: pd.Series, period: int) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def _wilder_adx(
    high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14
) -> pd.Series:
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()],
        axis=1,
    ).max(axis=1)
    up_move = high - prev_high
    down_move = prev_low - low
    plus_dm = pd.Series(
        np.where((up_move > down_move) & (up_move > 0.0), up_move, 0.0),
        index=high.index,
    )
    minus_dm = pd.Series(
        np.where((down_move > up_move) & (down_move > 0.0), down_move, 0.0),
        index=high.index,
    )
    alpha = 1.0 / float(period)
    atr_w = tr.ewm(alpha=alpha, min_periods=period, adjust=False).mean()
    atr_safe = atr_w.replace(0.0, np.nan)
    plus_di = 100.0 * plus_dm.ewm(
        alpha=alpha, min_periods=period, adjust=False
    ).mean() / atr_safe
    minus_di = 100.0 * minus_dm.ewm(
        alpha=alpha, min_periods=period, adjust=False
    ).mean() / atr_safe
    denom = (plus_di + minus_di).replace(0.0, np.nan)
    dx = 100.0 * (plus_di - minus_di).abs() / denom
    return dx.ewm(alpha=alpha, min_periods=period, adjust=False).mean()


def causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> pd.Series:
    """14-bar simple average of true range. Uses only bars at or before t."""
    close = df["close"].astype(np.float64)
    high = df["high"].astype(np.float64)
    low = df["low"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()],
        axis=1,
    ).max(axis=1)
    return tr.rolling(int(period), min_periods=int(period)).mean()


def add_fixed_ohlcv_features(
    df: pd.DataFrame, *, atr_period: int = ATR_PERIOD_DEFAULT
) -> pd.DataFrame:
    """Causal 50-feature matrix from OHLCV available at the close of bar t.

    Reads only ``time, open, high, low, close, volume``. Rolling windows use
    ``min_periods=window``. ATR is the simple 14-bar true-range mean used by
    the opportunity barriers.

    Args:
        df: OHLCV frame with a UTC ``time`` column.
        atr_period: True-range averaging window.

    Returns:
        Copy with ``atr`` plus ``FIXED_FEATURE_COLS`` appended.
    """
    _require_ohlcv(df)
    if len(FIXED_FEATURE_COLS) != 50:
        raise RuntimeError("FIXED_FEATURE_COLS must contain exactly 50 names")
    if len(set(FIXED_FEATURE_COLS)) != 50:
        raise RuntimeError("FIXED_FEATURE_COLS must be unique")

    out = df.loc[:, list(OHLCV_COLS)].copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    close_safe = close.replace(0.0, np.nan)
    atr = causal_atr(out, period=int(atr_period))
    out["atr"] = atr
    log_ret = np.log(close_safe).diff()
    log_volume = np.log1p(volume.clip(lower=0.0))
    body = close - open_
    body_sign = pd.Series(np.sign(body.to_numpy()), index=out.index).fillna(0.0)
    candle_range = high - low

    feats: Dict[str, pd.Series] = {}
    for k in (1, 2, 4, 8, 12, 24, 48):
        feats[f"return_{k}"] = np.log(close_safe / close_safe.shift(k))
    feats["candle_body"] = body / close_safe
    feats["range_pct"] = candle_range / close_safe
    feats["upper_wick_pct"] = (high - np.maximum(close, open_)) / close_safe
    feats["lower_wick_pct"] = (np.minimum(close, open_) - low) / close_safe
    feats["close_location"] = (close - low) / candle_range.replace(0.0, np.nan)

    ema_cache: Dict[int, pd.Series] = {}
    for span in (8, 16, 24, 48):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        ema_cache[span] = ema
        feats[f"ema_distance_{span}"] = (close - ema) / close_safe
    for span in (8, 24):
        ema = ema_cache[span]
        feats[f"ema_slope_{span}"] = (ema - ema.shift(1)) / close_safe

    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)

    ema12 = close.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = close.ewm(span=26, adjust=False, min_periods=26).mean()
    macd_raw = ema12 - ema26
    signal_raw = macd_raw.ewm(span=9, adjust=False, min_periods=9).mean()
    feats["macd"] = macd_raw / close_safe
    feats["macd_signal_distance"] = (macd_raw - signal_raw) / close_safe
    feats["adx_14"] = _wilder_adx(high, low, close, period=14)
    feats["atr_pct_14"] = atr / close_safe
    for window in (4, 12, 24):
        feats[f"rv_{window}"] = log_ret.rolling(window, min_periods=window).std()
    feats["volatility_ratio"] = feats["rv_4"] / feats["rv_24"].replace(0.0, np.nan)

    sma20 = close.rolling(20, min_periods=20).mean()
    std20 = close.rolling(20, min_periods=20).std()
    feats["bollinger_z_20"] = (close - sma20) / std20.replace(0.0, np.nan)

    for window in (24, 48):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_position_{window}"] = (close - ll) / (hh - ll).replace(
            0.0, np.nan
        )

    prior_high = high.shift(1).rolling(24, min_periods=24).max()
    prior_low = low.shift(1).rolling(24, min_periods=24).min()
    above = close - prior_high
    below = close - prior_low
    breakout = pd.Series(
        np.where(
            close > prior_high,
            above,
            np.where(close < prior_low, below, 0.0),
        ),
        index=out.index,
    )
    channel_ok = prior_high.notna() & prior_low.notna()
    feats["breakout_distance_24"] = (breakout / close_safe).where(channel_ok)

    feats["log_volume"] = log_volume
    feats["volume_change"] = log_volume.diff()
    vol_mean = volume.rolling(24, min_periods=24).mean()
    vol_std = volume.rolling(24, min_periods=24).std()
    feats["volume_zscore_24"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    feats["signed_volume_proxy"] = body_sign * log_volume
    feats["volume_trend_24"] = log_volume - log_volume.shift(24)

    changed = body_sign.ne(body_sign.shift())
    groups = changed.cumsum()
    feats["consecutive_candle_direction"] = body_sign * (
        body_sign.groupby(groups).cumcount() + 1
    )
    hh_flag = (high > high.shift(1)).astype(np.float64)
    ll_flag = (low < low.shift(1)).astype(np.float64)
    hh_flag.iloc[0] = np.nan
    ll_flag.iloc[0] = np.nan
    feats["higher_high_count_8"] = hh_flag.rolling(8, min_periods=8).sum()
    feats["lower_low_count_8"] = ll_flag.rolling(8, min_periods=8).sum()

    net = (close - close.shift(24)).abs()
    path = close.diff().abs().rolling(24, min_periods=24).sum()
    feats["trend_efficiency_24"] = net / path.replace(0.0, np.nan)
    feats["rolling_return_skew_24"] = log_ret.rolling(24, min_periods=24).skew()
    feats["rolling_return_kurtosis_24"] = log_ret.rolling(24, min_periods=24).kurt()

    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["weekday_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["weekday_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)

    hh48 = high.rolling(48, min_periods=48).max()
    ll48 = low.rolling(48, min_periods=48).min()
    feats["drawdown_48"] = close / hh48.replace(0.0, np.nan) - 1.0
    dist_high = (hh48 - close) / close_safe
    dist_low = (close - ll48) / close_safe
    feats["distance_from_rolling_extreme"] = pd.Series(
        np.minimum(dist_high.to_numpy(), dist_low.to_numpy()),
        index=out.index,
    )

    featured = pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)
    missing_cols = [c for c in FIXED_FEATURE_COLS if c not in featured.columns]
    if missing_cols:
        raise RuntimeError(f"Feature builder missing columns: {missing_cols}")
    return featured


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array; row j holds values shifted back by j+1 bars."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _first_touch_labels(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    tp_level: np.ndarray,
    sl_level: np.ndarray,
    *,
    short: bool,
) -> Tuple[np.ndarray, np.ndarray]:
    """First-touch TP/SL/timeout. Same-bar double touch is SL first."""
    if short:
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(fwd_high.shape[1])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    double_hit = has_hit & tp_first & sl_first
    opp = np.where(
        ~has_hit,
        OPP_TIMEOUT,
        np.where(sl_first, OPP_SL_FIRST, OPP_TP_FIRST),
    ).astype(np.int64)
    return opp, double_hit.astype(np.float64)


def attach_fixed_targets(
    df: pd.DataFrame,
    *,
    horizon_bars: int = FROZEN_HORIZON_BARS,
    tp_atr_mult: float = TP_ATR_MULT,
    sl_atr_mult: float = SL_ATR_MULT,
    dir_band_atr_mult: float = DIR_BAND_ATR_MULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Attach locked 24-bar targets. Last ``horizon_bars`` stay unlabeled.

    Entry is ``close[t]``. Long opportunity: TP 4 ATR above, SL 2 ATR below.
    Short opportunity: TP 4 ATR below, SL 2 ATR above. A same-bar double
    touch is labeled SL first on that side.

    Args:
        df: Frame with OHLCV and preferably a causal ``atr`` column.
        horizon_bars: Forward path length.
        tp_atr_mult: Take-profit distance in ATR units.
        sl_atr_mult: Stop distance in ATR units.
        dir_band_atr_mult: Neutral band around a zero 24-bar log return.
        atr_period: Used only when ``atr`` is missing.

    Returns:
        Copy with ``y_*`` targets, ``fwd_return_pct``, and causal ``tp_pct`` /
        ``sl_pct``.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    if "atr" in out.columns:
        atr = out["atr"].to_numpy(dtype=np.float64)
    else:
        atr = causal_atr(out, period=int(atr_period)).to_numpy(dtype=np.float64)

    y_dir = np.full(n, UNLABELED, dtype=np.int64)
    y_ret = np.full(n, np.nan, dtype=np.float64)
    y_vol = np.full(n, np.nan, dtype=np.float64)
    y_opp = np.full(n, UNLABELED, dtype=np.int64)
    y_opp_s = np.full(n, UNLABELED, dtype=np.int64)
    y_up = np.full(n, np.nan, dtype=np.float64)
    y_dn = np.full(n, np.nan, dtype=np.float64)
    double = np.full(n, np.nan, dtype=np.float64)
    double_s = np.full(n, np.nan, dtype=np.float64)
    fwd_pct = np.full(n, np.nan, dtype=np.float64)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close = _forward_stack(close, k)
    index = np.arange(n)
    horizon_ok = index < n - k
    atr_ok = np.isfinite(atr) & (atr > _EPS)
    close_ok = np.isfinite(close) & (np.abs(close) > _EPS)
    path_ok = np.isfinite(fwd_close).all(axis=0) & np.isfinite(fwd_high).all(
        axis=0
    ) & np.isfinite(fwd_low).all(axis=0)
    valid = horizon_ok & atr_ok & close_ok & path_ok

    end_close = np.full(n, np.nan, dtype=np.float64)
    if n > k:
        end_close[: n - k] = close[k:]
    with np.errstate(invalid="ignore", divide="ignore"):
        log_r = np.log(end_close / close)
        band = float(dir_band_atr_mult) * atr / close
        fwd_pct_all = end_close / close - 1.0
        prev = np.concatenate([close[None, :], fwd_close[:-1, :]], axis=0)
        step = np.log(fwd_close / prev)
        rms = np.sqrt(np.mean(np.square(step), axis=0))
        max_high = np.nanmax(
            np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0
        )
        min_low = np.nanmin(
            np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0
        )
        upside = (max_high - close) / close
        downside = (close - min_low) / close
        long_tp = close + float(tp_atr_mult) * atr
        long_sl = close - float(sl_atr_mult) * atr
        short_tp = close - float(tp_atr_mult) * atr
        short_sl = close + float(sl_atr_mult) * atr
        tp_pct_all = float(tp_atr_mult) * atr / close
        sl_pct_all = float(sl_atr_mult) * atr / close

    opp, double_hit = _first_touch_labels(
        fwd_high, fwd_low, long_tp, long_sl, short=False
    )
    opp_s, double_s_hit = _first_touch_labels(
        fwd_high, fwd_low, short_tp, short_sl, short=True
    )
    direction = np.where(
        log_r > band,
        DIR_UP,
        np.where(log_r < -band, DIR_DOWN, DIR_NEUTRAL),
    ).astype(np.int64)

    y_dir[valid] = direction[valid]
    y_ret[valid] = log_r[valid]
    y_vol[valid] = rms[valid]
    y_opp[valid] = opp[valid]
    y_opp_s[valid] = opp_s[valid]
    y_up[valid] = upside[valid]
    y_dn[valid] = downside[valid]
    double[valid] = double_hit[valid]
    double_s[valid] = double_s_hit[valid]
    fwd_pct[valid] = fwd_pct_all[valid]

    out["atr"] = atr
    out["y_direction"] = y_dir
    out["y_return"] = y_ret
    out["y_volatility"] = y_vol
    out["y_opportunity"] = y_opp
    out["y_opportunity_short"] = y_opp_s
    out["y_upside"] = y_up
    out["y_downside"] = y_dn
    out["same_bar_double_touch"] = double
    out["same_bar_double_touch_short"] = double_s
    out["fwd_return_pct"] = fwd_pct
    out["tp_pct"] = np.where(valid, tp_pct_all, np.nan)
    out["sl_pct"] = np.where(valid, sl_pct_all, np.nan)
    return out


def block_shuffle(
    values: np.ndarray, *, block: int, rng: np.random.Generator
) -> np.ndarray:
    """Permute labels in horizon-length blocks so serial dependence is kept."""
    arr = np.asarray(values)
    n = arr.shape[0]
    k = max(int(block), 1)
    parts = [arr[i : i + k] for i in range(0, n, k)]
    order = rng.permutation(len(parts))
    return np.concatenate([parts[i] for i in order], axis=0)[:n]


def shuffle_labeled_target(
    y: np.ndarray,
    *,
    labeled: np.ndarray,
    block: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Block-shuffle labeled rows only. Unlabeled positions stay unlabeled."""
    out = np.asarray(y).copy()
    idx = np.flatnonzero(np.asarray(labeled, dtype=bool))
    if idx.size == 0:
        return out
    out[idx] = block_shuffle(out[idx], block=block, rng=rng)
    return out


def _fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def _scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


def _xgb_kwargs(config: Mapping[str, Any], n_estimators: int, seed: int) -> Dict[str, Any]:
    return dict(
        n_estimators=int(n_estimators),
        max_depth=int(config["max_depth"]),
        learning_rate=float(config["learning_rate"]),
        min_child_weight=int(config["min_leaf"]),
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=5.0,
        tree_method="hist",
        n_jobs=int(config.get("threads_per_model", 2)),
        verbosity=0,
        random_state=int(seed),
    )


def _log_loss(y: np.ndarray, proba: np.ndarray, *, n_classes: int) -> float:
    y = np.asarray(y, dtype=np.int64).reshape(-1)
    p = np.asarray(proba, dtype=np.float64)
    if p.ndim != 2 or p.shape[0] != y.shape[0] or p.shape[1] != int(n_classes):
        return float("nan")
    if y.size == 0:
        return float("nan")
    p = np.clip(p, 1e-15, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    return float(-np.mean(np.log(p[np.arange(y.size), y])))


def _rmse(y: np.ndarray, pred: np.ndarray) -> float:
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    pred = np.asarray(pred, dtype=np.float64).reshape(-1)
    mask = np.isfinite(y) & np.isfinite(pred)
    if not mask.any():
        return float("nan")
    err = y[mask] - pred[mask]
    return float(np.sqrt(np.mean(np.square(err))))


def _class_prior(y: np.ndarray, n_classes: int) -> np.ndarray:
    prior = np.zeros(int(n_classes), dtype=np.float64)
    for c in range(int(n_classes)):
        prior[c] = float(np.mean(y == c))
    prior = np.clip(prior, 1e-6, 1.0)
    return prior / prior.sum()


def _align_multiclass_proba(clf: Any, x: np.ndarray, n_classes: int) -> np.ndarray:
    n = x.shape[0]
    out = np.full((n, int(n_classes)), 1.0 / float(n_classes), dtype=np.float64)
    if clf is None or n == 0:
        return out
    raw = np.asarray(clf.predict_proba(x), dtype=np.float64)
    classes = np.asarray(getattr(clf, "classes_", np.arange(raw.shape[1])))
    for j, cls in enumerate(classes.reshape(-1)):
        ci = int(cls)
        if 0 <= ci < int(n_classes) and j < raw.shape[1]:
            out[:, ci] = raw[:, j]
    out = np.clip(out, 1e-15, 1.0)
    return out / out.sum(axis=1, keepdims=True)


def majority_skill(model: Sequence[float], baseline: Sequence[float]) -> Dict[str, Any]:
    """True when a majority of scored folds beat the naive baseline (lower better)."""
    beats: List[bool] = []
    for m, b in zip(model, baseline):
        if np.isfinite(m) and np.isfinite(b):
            beats.append(bool(m < b))
    n = len(beats)
    share = float(np.mean(beats)) if n else float("nan")
    return {
        "n_folds_scored": n,
        "skill_share": share,
        "learnable": bool(n > 0 and share > 0.5),
    }


def _usable_class(frame: pd.DataFrame, y_col: str) -> np.ndarray:
    feats = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y = frame[y_col].to_numpy(dtype=np.int64)
    return (y >= 0) & np.isfinite(feats).all(axis=1)


def _usable_reg(frame: pd.DataFrame, y_col: str) -> np.ndarray:
    feats = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y = frame[y_col].to_numpy(dtype=np.float64)
    return np.isfinite(y) & np.isfinite(feats).all(axis=1)


def _in_split(n: int, split_idx: np.ndarray, usable: np.ndarray) -> np.ndarray:
    allowed = np.zeros(n, dtype=bool)
    idx = np.asarray(split_idx, dtype=np.int64)
    idx = idx[(idx >= 0) & (idx < n)]
    allowed[idx] = True
    return allowed & np.asarray(usable, dtype=bool)


def _class_mix(y: np.ndarray, names: Sequence[str]) -> Dict[str, float]:
    labeled = y[y >= 0]
    mix = {name: 0.0 for name in names}
    if labeled.size == 0:
        return mix
    for i, name in enumerate(names):
        mix[name] = float(np.mean(labeled == i))
    return mix


def _fit_classifier(
    x: np.ndarray,
    y: np.ndarray,
    *,
    config: Mapping[str, Any],
    n_estimators: int,
    seed: int,
) -> Any:
    from xgboost import XGBClassifier

    if len(np.unique(y)) < 2:
        return None
    clf = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        **_xgb_kwargs(config, n_estimators, seed),
    )
    clf.fit(x, y.astype(np.int32))
    return clf


def _fit_regressor(
    x: np.ndarray,
    y: np.ndarray,
    *,
    config: Mapping[str, Any],
    n_estimators: int,
    seed: int,
) -> Any:
    from xgboost import XGBRegressor

    if x.shape[0] < 8 or not np.isfinite(y).any():
        return None
    model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        **_xgb_kwargs(config, n_estimators, seed),
    )
    model.fit(x, y.astype(np.float32))
    return model


def _trade_pnl(
    pred: np.ndarray,
    fwd_pct: np.ndarray,
    *,
    cost: float,
    long_class: int,
    short_class: Optional[int] = None,
) -> Dict[str, float]:
    pnl = np.full(pred.shape[0], np.nan, dtype=np.float64)
    long_m = pred == int(long_class)
    pnl[long_m] = fwd_pct[long_m] - float(cost)
    if short_class is not None:
        short_m = pred == int(short_class)
        pnl[short_m] = -fwd_pct[short_m] - float(cost)
    taken = np.isfinite(pnl)
    if not taken.any():
        return {"trade_n": 0.0, "trade_mean_pnl": float("nan")}
    return {
        "trade_n": float(taken.sum()),
        "trade_mean_pnl": float(np.mean(pnl[taken])),
    }


def _walk_classifier(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
    y_override: Optional[np.ndarray] = None,
    score_true_y: bool = True,
) -> Dict[str, Any]:
    y_col = str(spec["y_col"])
    n_classes = int(spec["n_classes"])
    names = tuple(spec.get("names") or [str(i) for i in range(n_classes)])
    y_true = frame[y_col].to_numpy(dtype=np.int64)
    y_fit_all = y_true if y_override is None else np.asarray(y_override, dtype=np.int64)
    usable = _usable_class(frame, y_col)
    if y_override is not None:
        usable = usable & (y_fit_all >= 0)
    x_all = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    fwd = frame["fwd_return_pct"].to_numpy(dtype=np.float64)
    n = len(frame)
    model_ll: List[float] = []
    base_ll: List[float] = []
    trade_pnls: List[float] = []
    trade_ns: List[float] = []
    fold_rows: List[Dict[str, Any]] = []
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_m = _in_split(n, tr_idx, usable)
        eval_m = _in_split(n, va_idx, usable)
        if int(fit_m.sum()) < 32 or int(eval_m.sum()) < 8:
            continue
        y_tr = y_fit_all[fit_m]
        y_ev_fit = y_fit_all[eval_m]
        y_ev_true = y_true[eval_m]
        if len(np.unique(y_tr)) < 2:
            continue
        mean, std = _fit_scaler(x_all[fit_m])
        x_tr = _scale_x(x_all[fit_m], mean, std)
        x_ev = _scale_x(x_all[eval_m], mean, std)
        clf = _fit_classifier(
            x_tr,
            y_tr,
            config=config,
            n_estimators=n_estimators,
            seed=int(seed) + fold_i,
        )
        proba = _align_multiclass_proba(clf, x_ev, n_classes)
        y_score = y_ev_true if score_true_y else y_ev_fit
        valid_score = y_score >= 0
        if int(valid_score.sum()) < 8:
            continue
        ll = _log_loss(y_score[valid_score], proba[valid_score], n_classes=n_classes)
        prior = _class_prior(y_tr, n_classes)
        base = _log_loss(
            y_score[valid_score],
            np.repeat(prior[None, :], int(valid_score.sum()), axis=0),
            n_classes=n_classes,
        )
        pred = np.argmax(proba, axis=1)
        if spec["id"] == "direction_head":
            traded = _trade_pnl(
                pred, fwd[eval_m], cost=float(config["round_trip_cost_pct"]),
                long_class=DIR_UP, short_class=DIR_DOWN,
            )
        else:
            traded = _trade_pnl(
                pred, fwd[eval_m], cost=float(config["round_trip_cost_pct"]),
                long_class=OPP_TP_FIRST, short_class=None,
            )
        model_ll.append(ll)
        base_ll.append(base)
        trade_pnls.append(float(traded["trade_mean_pnl"]))
        trade_ns.append(float(traded["trade_n"]))
        fold_rows.append(
            {
                "fold": fold_i,
                "log_loss": ll,
                "baseline_log_loss": base,
                "skill": bool(np.isfinite(ll) and np.isfinite(base) and ll < base),
                **traded,
            }
        )
        print(
            f"{spec['id']} fold {fold_i}: log_loss={ll:.4f} baseline={base:.4f} "
            f"skill={fold_rows[-1]['skill']} trades={traded['trade_n']:.0f}"
        )
    gate = majority_skill(model_ll, base_ll)
    finite_pnl = [p for p in trade_pnls if np.isfinite(p)]
    return {
        "head": spec["id"],
        "kind": "classification",
        "n_classes": n_classes,
        "class_mix": _class_mix(y_true, names),
        "wf_log_loss": float(np.mean(model_ll)) if model_ll else float("nan"),
        "wf_baseline_log_loss": float(np.mean(base_ll)) if base_ll else float("nan"),
        "trade_mean_pnl": float(np.mean(finite_pnl)) if finite_pnl else float("nan"),
        "trade_n": float(np.sum(trade_ns)) if trade_ns else 0.0,
        "fold_rows": fold_rows,
        **gate,
    }


def _walk_regressor(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    y_col = str(spec["y_col"])
    usable = _usable_reg(frame, y_col)
    x_all = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y_all = frame[y_col].to_numpy(dtype=np.float64)
    n = len(frame)
    model_s: List[float] = []
    base_s: List[float] = []
    fold_rows: List[Dict[str, Any]] = []
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_m = _in_split(n, tr_idx, usable)
        eval_m = _in_split(n, va_idx, usable)
        if int(fit_m.sum()) < 32 or int(eval_m.sum()) < 8:
            continue
        mean, std = _fit_scaler(x_all[fit_m])
        model = _fit_regressor(
            _scale_x(x_all[fit_m], mean, std),
            y_all[fit_m],
            config=config,
            n_estimators=n_estimators,
            seed=int(seed) + fold_i,
        )
        if model is None:
            continue
        pred = np.asarray(
            model.predict(_scale_x(x_all[eval_m], mean, std)), dtype=np.float64
        )
        rmse = _rmse(y_all[eval_m], pred)
        base = _rmse(y_all[eval_m], np.full(int(eval_m.sum()), y_all[fit_m].mean()))
        model_s.append(rmse)
        base_s.append(base)
        fold_rows.append(
            {
                "fold": fold_i,
                "rmse": rmse,
                "baseline_rmse": base,
                "skill": bool(np.isfinite(rmse) and np.isfinite(base) and rmse < base),
            }
        )
        print(
            f"{spec['id']} fold {fold_i}: rmse={rmse:.6f} baseline={base:.6f} "
            f"skill={fold_rows[-1]['skill']}"
        )
    gate = majority_skill(model_s, base_s)
    return {
        "head": spec["id"],
        "kind": "regression",
        "wf_rmse": float(np.mean(model_s)) if model_s else float("nan"),
        "wf_baseline_rmse": float(np.mean(base_s)) if base_s else float("nan"),
        "fold_rows": fold_rows,
        **gate,
    }


def _walk_excursion(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    y_up_col, y_dn_col = spec["y_cols"]
    usable = _usable_reg(frame, y_up_col) & _usable_reg(frame, y_dn_col)
    x_all = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y_up = frame[str(y_up_col)].to_numpy(dtype=np.float64)
    y_dn = frame[str(y_dn_col)].to_numpy(dtype=np.float64)
    n = len(frame)
    up_m: List[float] = []
    up_b: List[float] = []
    dn_m: List[float] = []
    dn_b: List[float] = []
    fold_rows: List[Dict[str, Any]] = []
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_m = _in_split(n, tr_idx, usable)
        eval_m = _in_split(n, va_idx, usable)
        if int(fit_m.sum()) < 32 or int(eval_m.sum()) < 8:
            continue
        mean, std = _fit_scaler(x_all[fit_m])
        x_tr = _scale_x(x_all[fit_m], mean, std)
        x_ev = _scale_x(x_all[eval_m], mean, std)
        up_model = _fit_regressor(
            x_tr, y_up[fit_m], config=config, n_estimators=n_estimators,
            seed=int(seed) + fold_i,
        )
        dn_model = _fit_regressor(
            x_tr, y_dn[fit_m], config=config, n_estimators=n_estimators,
            seed=int(seed) + fold_i + 17,
        )
        if up_model is None or dn_model is None:
            continue
        up_pred = np.asarray(up_model.predict(x_ev), dtype=np.float64)
        dn_pred = np.asarray(dn_model.predict(x_ev), dtype=np.float64)
        up_rmse = _rmse(y_up[eval_m], up_pred)
        dn_rmse = _rmse(y_dn[eval_m], dn_pred)
        up_base = _rmse(y_up[eval_m], np.full(int(eval_m.sum()), y_up[fit_m].mean()))
        dn_base = _rmse(y_dn[eval_m], np.full(int(eval_m.sum()), y_dn[fit_m].mean()))
        up_m.append(up_rmse)
        up_b.append(up_base)
        dn_m.append(dn_rmse)
        dn_b.append(dn_base)
        both = (
            np.isfinite(up_rmse)
            and np.isfinite(dn_rmse)
            and up_rmse < up_base
            and dn_rmse < dn_base
        )
        fold_rows.append(
            {
                "fold": fold_i,
                "rmse_upside": up_rmse,
                "rmse_downside": dn_rmse,
                "baseline_rmse_upside": up_base,
                "baseline_rmse_downside": dn_base,
                "skill": bool(both),
            }
        )
        print(
            f"{spec['id']} fold {fold_i}: up={up_rmse:.6f}/{up_base:.6f} "
            f"dn={dn_rmse:.6f}/{dn_base:.6f} skill={both}"
        )
    both_share = (
        float(np.mean([bool(r["skill"]) for r in fold_rows])) if fold_rows else float("nan")
    )
    n_scored = len(fold_rows)
    return {
        "head": spec["id"],
        "kind": "regression_pair",
        "wf_rmse_upside": float(np.mean(up_m)) if up_m else float("nan"),
        "wf_rmse_downside": float(np.mean(dn_m)) if dn_m else float("nan"),
        "wf_baseline_rmse_upside": float(np.mean(up_b)) if up_b else float("nan"),
        "wf_baseline_rmse_downside": float(np.mean(dn_b)) if dn_b else float("nan"),
        "n_folds_scored": n_scored,
        "skill_share": both_share,
        "learnable": bool(n_scored > 0 and both_share > 0.5),
        "fold_rows": fold_rows,
    }


def _confirm_classifier_test(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    splits: Mapping[str, np.ndarray],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    y_col = str(spec["y_col"])
    n_classes = int(spec["n_classes"])
    usable = _usable_class(frame, y_col)
    x_all = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y_all = frame[y_col].to_numpy(dtype=np.int64)
    fwd = frame["fwd_return_pct"].to_numpy(dtype=np.float64)
    n = len(frame)
    train_idx = np.concatenate(
        [np.asarray(splits["train"]), np.asarray(splits["val"])], axis=0
    )
    fit_m = _in_split(n, train_idx, usable)
    test_m = _in_split(n, splits["test"], usable)
    if int(fit_m.sum()) < 32 or int(test_m.sum()) < 8 or len(np.unique(y_all[fit_m])) < 2:
        return {"head": spec["id"], "test_scored": False}
    mean, std = _fit_scaler(x_all[fit_m])
    clf = _fit_classifier(
        _scale_x(x_all[fit_m], mean, std),
        y_all[fit_m],
        config=config,
        n_estimators=n_estimators,
        seed=seed,
    )
    proba = _align_multiclass_proba(clf, _scale_x(x_all[test_m], mean, std), n_classes)
    ll = _log_loss(y_all[test_m], proba, n_classes=n_classes)
    prior = _class_prior(y_all[fit_m], n_classes)
    base = _log_loss(
        y_all[test_m],
        np.repeat(prior[None, :], int(test_m.sum()), axis=0),
        n_classes=n_classes,
    )
    pred = np.argmax(proba, axis=1)
    if spec["id"] == "direction_head":
        traded = _trade_pnl(
            pred, fwd[test_m], cost=float(config["round_trip_cost_pct"]),
            long_class=DIR_UP, short_class=DIR_DOWN,
        )
    else:
        traded = _trade_pnl(
            pred, fwd[test_m], cost=float(config["round_trip_cost_pct"]),
            long_class=OPP_TP_FIRST, short_class=None,
        )
    return {
        "head": spec["id"],
        "test_scored": True,
        "test_log_loss": ll,
        "test_baseline_log_loss": base,
        "test_skill": bool(np.isfinite(ll) and np.isfinite(base) and ll < base),
        "n_test": float(test_m.sum()),
        **{f"test_{k}": v for k, v in traded.items()},
    }


def _confirm_regressor_test(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    splits: Mapping[str, np.ndarray],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    y_col = str(spec["y_col"])
    usable = _usable_reg(frame, y_col)
    x_all = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y_all = frame[y_col].to_numpy(dtype=np.float64)
    n = len(frame)
    train_idx = np.concatenate(
        [np.asarray(splits["train"]), np.asarray(splits["val"])], axis=0
    )
    fit_m = _in_split(n, train_idx, usable)
    test_m = _in_split(n, splits["test"], usable)
    if int(fit_m.sum()) < 32 or int(test_m.sum()) < 8:
        return {"head": spec["id"], "test_scored": False}
    mean, std = _fit_scaler(x_all[fit_m])
    model = _fit_regressor(
        _scale_x(x_all[fit_m], mean, std),
        y_all[fit_m],
        config=config,
        n_estimators=n_estimators,
        seed=seed,
    )
    if model is None:
        return {"head": spec["id"], "test_scored": False}
    pred = np.asarray(
        model.predict(_scale_x(x_all[test_m], mean, std)), dtype=np.float64
    )
    rmse = _rmse(y_all[test_m], pred)
    base = _rmse(y_all[test_m], np.full(int(test_m.sum()), y_all[fit_m].mean()))
    return {
        "head": spec["id"],
        "test_scored": True,
        "test_rmse": rmse,
        "test_baseline_rmse": base,
        "test_skill": bool(np.isfinite(rmse) and np.isfinite(base) and rmse < base),
        "n_test": float(test_m.sum()),
    }


def _confirm_excursion_test(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    splits: Mapping[str, np.ndarray],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    y_up_col, y_dn_col = spec["y_cols"]
    usable = _usable_reg(frame, y_up_col) & _usable_reg(frame, y_dn_col)
    x_all = frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64)
    y_up = frame[str(y_up_col)].to_numpy(dtype=np.float64)
    y_dn = frame[str(y_dn_col)].to_numpy(dtype=np.float64)
    n = len(frame)
    train_idx = np.concatenate(
        [np.asarray(splits["train"]), np.asarray(splits["val"])], axis=0
    )
    fit_m = _in_split(n, train_idx, usable)
    test_m = _in_split(n, splits["test"], usable)
    if int(fit_m.sum()) < 32 or int(test_m.sum()) < 8:
        return {"head": spec["id"], "test_scored": False}
    mean, std = _fit_scaler(x_all[fit_m])
    x_tr = _scale_x(x_all[fit_m], mean, std)
    x_te = _scale_x(x_all[test_m], mean, std)
    up_model = _fit_regressor(
        x_tr, y_up[fit_m], config=config, n_estimators=n_estimators, seed=seed
    )
    dn_model = _fit_regressor(
        x_tr, y_dn[fit_m], config=config, n_estimators=n_estimators, seed=seed + 17
    )
    if up_model is None or dn_model is None:
        return {"head": spec["id"], "test_scored": False}
    up_rmse = _rmse(y_up[test_m], np.asarray(up_model.predict(x_te), dtype=np.float64))
    dn_rmse = _rmse(y_dn[test_m], np.asarray(dn_model.predict(x_te), dtype=np.float64))
    up_base = _rmse(y_up[test_m], np.full(int(test_m.sum()), y_up[fit_m].mean()))
    dn_base = _rmse(y_dn[test_m], np.full(int(test_m.sum()), y_dn[fit_m].mean()))
    return {
        "head": spec["id"],
        "test_scored": True,
        "test_rmse_upside": up_rmse,
        "test_rmse_downside": dn_rmse,
        "test_baseline_rmse_upside": up_base,
        "test_baseline_rmse_downside": dn_base,
        "test_skill": bool(
            np.isfinite(up_rmse)
            and np.isfinite(dn_rmse)
            and up_rmse < up_base
            and dn_rmse < dn_base
        ),
        "n_test": float(test_m.sum()),
    }


def _json_sanitize(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): _json_sanitize(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_sanitize(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return _json_sanitize(obj.tolist())
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return value if np.isfinite(value) else None
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    return obj


def _print_stage_frame(name: str, rows: Sequence[Mapping[str, Any]]) -> None:
    print(f"\n===== {name} =====")
    if not rows:
        print("(empty)")
        return
    frame = pd.DataFrame([dict(r) for r in rows])
    nested = [
        c
        for c in frame.columns
        if frame[c].map(lambda v: isinstance(v, (dict, list))).any()
    ]
    show = frame.drop(columns=nested) if nested else frame
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None, "display.width", 220
    ):
        print(show.to_string(index=False))


def _walk_one(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    kind = str(spec["kind"])
    if kind == "classification":
        return _walk_classifier(
            frame, spec, wf_folds, config, n_estimators=n_estimators, seed=seed
        )
    if kind == "regression":
        return _walk_regressor(
            frame, spec, wf_folds, config, n_estimators=n_estimators, seed=seed
        )
    return _walk_excursion(
        frame, spec, wf_folds, config, n_estimators=n_estimators, seed=seed
    )


def _confirm_one(
    frame: pd.DataFrame,
    spec: Mapping[str, Any],
    splits: Mapping[str, np.ndarray],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    kind = str(spec["kind"])
    if kind == "classification":
        return _confirm_classifier_test(
            frame, spec, splits, config, n_estimators=n_estimators, seed=seed
        )
    if kind == "regression":
        return _confirm_regressor_test(
            frame, spec, splits, config, n_estimators=n_estimators, seed=seed
        )
    return _confirm_excursion_test(
        frame, spec, splits, config, n_estimators=n_estimators, seed=seed
    )


def run_fixed_target_experiments(
    bars: pd.DataFrame,
    *,
    config: Mapping[str, Any],
    splits: Mapping[str, np.ndarray],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    export_dir: Path,
    n_estimators: Optional[int] = None,
) -> Dict[str, Any]:
    """Train each locked target separately on the frozen 50-feature set.

    Learnability is walk-forward skill versus a naive baseline. The held-out
    test split is scored once for heads that pass. Classifiers also get a
    block-shuffled label null.
    """
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except Exception:
        pass
    trees = int(n_estimators or config.get("n_estimators", N_ESTIMATORS_DEFAULT))
    seed = int(config.get("seed", SCREEN_SEED))
    rng = np.random.default_rng(seed)
    featured = add_fixed_ohlcv_features(
        bars, atr_period=int(config.get("atr_period", ATR_PERIOD_DEFAULT))
    )
    labeled = attach_fixed_targets(
        featured,
        horizon_bars=int(config.get("horizon_bars", FROZEN_HORIZON_BARS)),
        tp_atr_mult=TP_ATR_MULT,
        sl_atr_mult=SL_ATR_MULT,
        dir_band_atr_mult=DIR_BAND_ATR_MULT,
        atr_period=int(config.get("atr_period", ATR_PERIOD_DEFAULT)),
    )
    null_rates = labeled.loc[:, list(FIXED_FEATURE_COLS)].isna().mean()
    print("n_features", len(FIXED_FEATURE_COLS))
    print(
        "max feature null rate",
        round(float(null_rates.max()), 4),
        "at",
        str(null_rates.idxmax()),
    )
    double_n = int(np.nansum(labeled["same_bar_double_touch"].to_numpy()))
    print("opportunity same-bar double-touch", double_n)

    walk_forward: List[Dict[str, Any]] = []
    shuffle_rows: List[Dict[str, Any]] = []
    test_rows: List[Dict[str, Any]] = []
    spec_by_id = {str(s["id"]): s for s in TARGET_SPECS}
    for spec in TARGET_SPECS:
        print(f"\n===== {spec['id']} =====")
        result = _walk_one(
            labeled, spec, wf_folds, config, n_estimators=trees, seed=seed
        )
        walk_forward.append(result)
        if spec["kind"] == "classification":
            y_col = str(spec["y_col"])
            usable = _usable_class(labeled, y_col)
            shuffled = shuffle_labeled_target(
                labeled[y_col].to_numpy(dtype=np.int64),
                labeled=usable,
                block=int(config.get("horizon_bars", FROZEN_HORIZON_BARS)),
                rng=rng,
            )
            null = _walk_classifier(
                labeled,
                spec,
                wf_folds,
                config,
                n_estimators=trees,
                seed=seed + 100,
                y_override=shuffled,
                score_true_y=True,
            )
            shuffle_rows.append(
                {
                    "head": spec["id"],
                    "wf_log_loss": null["wf_log_loss"],
                    "wf_baseline_log_loss": null["wf_baseline_log_loss"],
                    "skill_share": null["skill_share"],
                    "learnable": null["learnable"],
                }
            )
        if result.get("learnable"):
            print(f"Test once {spec['id']}")
            test_rows.append(
                _confirm_one(
                    labeled, spec, splits, config, n_estimators=trees, seed=seed
                )
            )

    learnable = [r["head"] for r in walk_forward if r.get("learnable")]
    summary_rows = []
    for row in walk_forward:
        summary_rows.append(
            {
                "head": row["head"],
                "kind": row["kind"],
                "learnable": row.get("learnable"),
                "n_folds_scored": row.get("n_folds_scored"),
                "skill_share": row.get("skill_share"),
                "wf_log_loss": row.get("wf_log_loss"),
                "wf_baseline_log_loss": row.get("wf_baseline_log_loss"),
                "wf_rmse": row.get("wf_rmse"),
                "wf_baseline_rmse": row.get("wf_baseline_rmse"),
                "wf_rmse_upside": row.get("wf_rmse_upside"),
                "wf_rmse_downside": row.get("wf_rmse_downside"),
                "trade_mean_pnl": row.get("trade_mean_pnl"),
                "trade_n": row.get("trade_n"),
                "class_mix": row.get("class_mix"),
            }
        )
    _print_stage_frame("Walk-forward skill", summary_rows)
    _print_stage_frame("Shuffle null", shuffle_rows)
    _print_stage_frame("Test once", test_rows)
    print("learnable", learnable)
    print("opportunity_double_touch_n", double_n)

    payload = {
        "resolution": str(config.get("resolution", FROZEN_RESOLUTION)),
        "horizon_bars": int(config.get("horizon_bars", FROZEN_HORIZON_BARS)),
        "model": "XGB",
        "n_estimators": trees,
        "n_features": len(FIXED_FEATURE_COLS),
        "feature_cols": list(FIXED_FEATURE_COLS),
        "tp_atr_mult": TP_ATR_MULT,
        "sl_atr_mult": SL_ATR_MULT,
        "dir_band_atr_mult": DIR_BAND_ATR_MULT,
        "entry": "close[t]",
        "opportunity_side": "long",
        "opportunity_double_touch_n": double_n,
        "walk_forward": walk_forward,
        "shuffle_rows": shuffle_rows,
        "test_rows": test_rows,
        "learnable": learnable,
        "target_ids": [s["id"] for s in TARGET_SPECS],
        "spec_by_id": {k: {"id": v["id"], "kind": v["kind"]} for k, v in spec_by_id.items()},
    }
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)
    (export_dir / "sweep.json").write_text(
        json.dumps(_json_sanitize(payload), indent=2),
        encoding="utf-8",
    )
    print("wrote", export_dir / "sweep.json")
    return payload


def long_path_pnl(
    y_opp: np.ndarray,
    tp_pct: np.ndarray,
    sl_pct: np.ndarray,
    timeout_ret: np.ndarray,
    *,
    cost: float,
) -> np.ndarray:
    """Gross-of-path long P&L minus round-trip cost."""
    y = np.asarray(y_opp, dtype=np.int64)
    tp = np.asarray(tp_pct, dtype=np.float64)
    sl = np.asarray(sl_pct, dtype=np.float64)
    timeout = np.asarray(timeout_ret, dtype=np.float64)
    gross = np.where(
        y == OPP_TP_FIRST,
        tp,
        np.where(y == OPP_SL_FIRST, -sl, timeout),
    )
    return gross - float(cost)


def short_path_pnl(
    y_opp: np.ndarray,
    tp_pct: np.ndarray,
    sl_pct: np.ndarray,
    timeout_ret: np.ndarray,
    *,
    cost: float,
) -> np.ndarray:
    """Gross-of-path short P&L minus round-trip cost."""
    y = np.asarray(y_opp, dtype=np.int64)
    tp = np.asarray(tp_pct, dtype=np.float64)
    sl = np.asarray(sl_pct, dtype=np.float64)
    timeout = np.asarray(timeout_ret, dtype=np.float64)
    gross = np.where(
        y == OPP_TP_FIRST,
        tp,
        np.where(y == OPP_SL_FIRST, -sl, -timeout),
    )
    return gross - float(cost)


def r_multiple(pnl: np.ndarray, sl_pct: np.ndarray) -> np.ndarray:
    """P&L in stop-distance units. Causal sl_pct only."""
    sl = np.asarray(sl_pct, dtype=np.float64)
    pnl_arr = np.asarray(pnl, dtype=np.float64)
    return pnl_arr / np.where(np.abs(sl) > R_EPS, sl, np.nan)


def apply_policy_mask(
    p_tp: np.ndarray,
    pred_vol: np.ndarray,
    pred_down: np.ndarray,
    tp_pct: np.ndarray,
    sl_pct: np.ndarray,
    *,
    thr_opp: float,
    k_vol: float,
    k_dd: float,
) -> np.ndarray:
    """Take a side when TP-prob, vol, and adverse excursion clear filters."""
    tp = np.asarray(tp_pct, dtype=np.float64)
    sl = np.asarray(sl_pct, dtype=np.float64)
    return (
        (np.asarray(p_tp, dtype=np.float64) >= float(thr_opp))
        & (np.asarray(pred_vol, dtype=np.float64) >= float(k_vol) * tp)
        & (np.asarray(pred_down, dtype=np.float64) <= float(k_dd) * sl)
        & np.isfinite(p_tp)
        & np.isfinite(pred_vol)
        & np.isfinite(pred_down)
        & np.isfinite(tp)
        & np.isfinite(sl)
    )


def score_policy_r(
    take: np.ndarray,
    pnl: np.ndarray,
    sl_pct: np.ndarray,
    min_signals: int,
) -> Dict[str, float]:
    """R-expectancy on taken trades. No fallback when n < min_signals."""
    mask = np.asarray(take, dtype=bool).reshape(-1)
    r = r_multiple(pnl[mask], sl_pct[mask])
    r = r[np.isfinite(r)]
    n = int(r.size)
    if n < int(min_signals):
        return {
            "r_expectancy": float("-inf"),
            "tstat": float("nan"),
            "n": float(n),
        }
    mean_r = float(r.mean())
    std = float(r.std(ddof=1)) if n > 1 else float("nan")
    tstat = (
        float(mean_r / (std / np.sqrt(n))) if n > 1 and std > 0 else float("nan")
    )
    return {"r_expectancy": mean_r, "tstat": tstat, "n": float(n)}


def apply_twosided_policy(
    p_tp_long: np.ndarray,
    p_tp_short: np.ndarray,
    pred_vol: np.ndarray,
    pred_up: np.ndarray,
    pred_down: np.ndarray,
    tp_pct: np.ndarray,
    sl_pct: np.ndarray,
    *,
    thr_long: float,
    thr_short: float,
    k_vol: float,
    k_dd: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Long and short masks. Both-true bars are flattened."""
    long_raw = apply_policy_mask(
        p_tp_long,
        pred_vol,
        pred_down,
        tp_pct,
        sl_pct,
        thr_opp=thr_long,
        k_vol=k_vol,
        k_dd=k_dd,
    )
    short_raw = apply_policy_mask(
        p_tp_short,
        pred_vol,
        pred_up,
        tp_pct,
        sl_pct,
        thr_opp=thr_short,
        k_vol=k_vol,
        k_dd=k_dd,
    )
    conflict = long_raw & short_raw
    return long_raw & ~conflict, short_raw & ~conflict


def score_twosided_r(
    long_take: np.ndarray,
    short_take: np.ndarray,
    pnl_long: np.ndarray,
    pnl_short: np.ndarray,
    sl_pct: np.ndarray,
    min_signals: int,
) -> Dict[str, float]:
    """Combined R plus per-side R. Conflicts must already be removed."""
    long_m = np.asarray(long_take, dtype=bool).reshape(-1)
    short_m = np.asarray(short_take, dtype=bool).reshape(-1)
    pnl = np.where(long_m, pnl_long, np.where(short_m, pnl_short, np.nan))
    total = score_policy_r(long_m | short_m, pnl, sl_pct, min_signals)
    long_s = score_policy_r(long_m, pnl_long, sl_pct, min_signals)
    short_s = score_policy_r(short_m, pnl_short, sl_pct, min_signals)
    return {
        "r_expectancy": total["r_expectancy"],
        "tstat": total["tstat"],
        "n": total["n"],
        "long_r_expectancy": long_s["r_expectancy"],
        "long_n": long_s["n"],
        "short_r_expectancy": short_s["r_expectancy"],
        "short_n": short_s["n"],
    }


def _mean_finite_r(
    rows: Sequence[Mapping[str, Any]], key: str = "r_expectancy"
) -> float:
    vals = [
        float(r[key])
        for r in rows
        if key in r and np.isfinite(r[key]) and r[key] > float("-inf") / 2
    ]
    return float(np.mean(vals)) if vals else float("-inf")


def choose_policy_thresholds(
    p_tp: np.ndarray,
    pred_vol: np.ndarray,
    pred_down: np.ndarray,
    tp_pct: np.ndarray,
    sl_pct: np.ndarray,
    pnl: np.ndarray,
    *,
    min_signals: int,
    opp_quantiles: Sequence[float] = OPP_QUANTILES,
    k_vol_grid: Sequence[float] = K_VOL_GRID,
    k_dd_grid: Sequence[float] = K_DD_GRID,
) -> Dict[str, float]:
    """Pick filter thresholds on calibration rows by R-expectancy."""
    best: Dict[str, float] = {
        "thr_opp": 1.0,
        "k_vol": float(k_vol_grid[-1]),
        "k_dd": float(k_dd_grid[0]),
        "r_expectancy": float("-inf"),
        "n": 0.0,
    }
    finite = np.isfinite(p_tp)
    if not finite.any():
        return best
    for q in opp_quantiles:
        thr = float(np.quantile(p_tp[finite], float(q)))
        for k_vol in k_vol_grid:
            for k_dd in k_dd_grid:
                take = apply_policy_mask(
                    p_tp,
                    pred_vol,
                    pred_down,
                    tp_pct,
                    sl_pct,
                    thr_opp=thr,
                    k_vol=float(k_vol),
                    k_dd=float(k_dd),
                )
                scored = score_policy_r(take, pnl, sl_pct, min_signals)
                if float(scored["r_expectancy"]) > float(best["r_expectancy"]):
                    best = {
                        "thr_opp": thr,
                        "k_vol": float(k_vol),
                        "k_dd": float(k_dd),
                        "r_expectancy": float(scored["r_expectancy"]),
                        "n": float(scored["n"]),
                    }
    return best


def choose_twosided_thresholds(
    p_tp_long: np.ndarray,
    p_tp_short: np.ndarray,
    pred_vol: np.ndarray,
    pred_up: np.ndarray,
    pred_down: np.ndarray,
    tp_pct: np.ndarray,
    sl_pct: np.ndarray,
    pnl_long: np.ndarray,
    pnl_short: np.ndarray,
    *,
    min_signals: int,
    opp_quantiles: Sequence[float] = OPP_QUANTILES,
    k_vol_grid: Sequence[float] = K_VOL_GRID,
    k_dd_grid: Sequence[float] = K_DD_GRID,
) -> Dict[str, float]:
    """Pick long/short filters on calibration rows by combined R."""
    best: Dict[str, float] = {
        "thr_long": 1.0,
        "thr_short": 1.0,
        "k_vol": float(k_vol_grid[-1]),
        "k_dd": float(k_dd_grid[0]),
        "r_expectancy": float("-inf"),
        "n": 0.0,
        "long_n": 0.0,
        "short_n": 0.0,
    }
    finite_l = np.isfinite(p_tp_long)
    finite_s = np.isfinite(p_tp_short)
    if not finite_l.any() or not finite_s.any():
        return best
    for q_l in opp_quantiles:
        thr_l = float(np.quantile(p_tp_long[finite_l], float(q_l)))
        for q_s in opp_quantiles:
            thr_s = float(np.quantile(p_tp_short[finite_s], float(q_s)))
            for k_vol in k_vol_grid:
                for k_dd in k_dd_grid:
                    long_t, short_t = apply_twosided_policy(
                        p_tp_long,
                        p_tp_short,
                        pred_vol,
                        pred_up,
                        pred_down,
                        tp_pct,
                        sl_pct,
                        thr_long=thr_l,
                        thr_short=thr_s,
                        k_vol=float(k_vol),
                        k_dd=float(k_dd),
                    )
                    scored = score_twosided_r(
                        long_t,
                        short_t,
                        pnl_long,
                        pnl_short,
                        sl_pct,
                        min_signals,
                    )
                    if float(scored["r_expectancy"]) > float(best["r_expectancy"]):
                        best = {
                            "thr_long": thr_l,
                            "thr_short": thr_s,
                            "k_vol": float(k_vol),
                            "k_dd": float(k_dd),
                            "r_expectancy": float(scored["r_expectancy"]),
                            "n": float(scored["n"]),
                            "long_n": float(scored["long_n"]),
                            "short_n": float(scored["short_n"]),
                        }
    return best


def policy_test_gate(
    fold_r: Sequence[float],
    unfiltered_r: float,
    shuffle_r: float,
    *,
    long_r: Optional[float] = None,
    short_r: Optional[float] = None,
    long_n: float = 0.0,
    short_n: float = 0.0,
    min_signals: int = MIN_SIGNALS_DEFAULT,
) -> bool:
    """True when mean WF R is positive, both sides pay, and beats comparators."""
    finite = [float(v) for v in fold_r if np.isfinite(v) and v > float("-inf") / 2]
    if not finite:
        return False
    mean_r = float(np.mean(finite))
    pos_share = float(np.mean([v > 0.0 for v in finite]))
    if mean_r <= 0.0 or pos_share <= 0.5:
        return False
    if np.isfinite(unfiltered_r) and mean_r <= float(unfiltered_r):
        return False
    if np.isfinite(shuffle_r) and mean_r <= float(shuffle_r):
        return False
    if long_r is None and short_r is None:
        return True
    if float(long_n) < int(min_signals) or float(short_n) < int(min_signals):
        return False
    if not (np.isfinite(long_r) and float(long_r) > 0.0):
        return False
    if not (np.isfinite(short_r) and float(short_r) > 0.0):
        return False
    return True


def _usable_policy(frame: pd.DataFrame) -> np.ndarray:
    return (
        _usable_class(frame, "y_opportunity")
        & _usable_class(frame, "y_opportunity_short")
        & _usable_reg(frame, "y_volatility")
        & _usable_reg(frame, "y_upside")
        & _usable_reg(frame, "y_downside")
        & np.isfinite(frame["tp_pct"].to_numpy(dtype=np.float64))
        & np.isfinite(frame["sl_pct"].to_numpy(dtype=np.float64))
    )


def inner_split_train(
    train_idx: np.ndarray,
    *,
    calib_fraction: float,
    purge_bars: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Purged tail of a training window for threshold selection."""
    idx = np.asarray(train_idx, dtype=np.int64)
    n = idx.shape[0]
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = idx[: max(cut - int(purge_bars), 0)]
    calib_idx = idx[cut:]
    return fit_idx, calib_idx


def torch_available() -> bool:
    """True when PyTorch can be imported."""
    try:
        import torch  # noqa: F401

        return True
    except Exception:
        return False


def _fit_xgb_policy_heads(
    x: np.ndarray,
    y_opp: np.ndarray,
    y_opp_short: np.ndarray,
    y_vol: np.ndarray,
    y_up: np.ndarray,
    y_dn: np.ndarray,
    *,
    config: Mapping[str, Any],
    n_estimators: int,
    seed: int,
) -> Dict[str, Any]:
    opp = _fit_classifier(
        x, y_opp, config=config, n_estimators=n_estimators, seed=seed
    )
    opp_s = _fit_classifier(
        x, y_opp_short, config=config, n_estimators=n_estimators, seed=seed + 1
    )
    vol = _fit_regressor(
        x, y_vol, config=config, n_estimators=n_estimators, seed=seed + 2
    )
    up = _fit_regressor(
        x, y_up, config=config, n_estimators=n_estimators, seed=seed + 3
    )
    dn = _fit_regressor(
        x, y_dn, config=config, n_estimators=n_estimators, seed=seed + 4
    )
    return {
        "kind": "xgb",
        "opp": opp,
        "opp_short": opp_s,
        "vol": vol,
        "up": up,
        "dn": dn,
    }


def _predict_xgb_heads(
    models: Mapping[str, Any], x: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = x.shape[0]
    p_long = _align_multiclass_proba(models.get("opp"), x, 3)
    p_short = _align_multiclass_proba(models.get("opp_short"), x, 3)
    vol = (
        np.asarray(models["vol"].predict(x), dtype=np.float64)
        if models.get("vol") is not None
        else np.zeros(n, dtype=np.float64)
    )
    up = (
        np.asarray(models["up"].predict(x), dtype=np.float64)
        if models.get("up") is not None
        else np.zeros(n, dtype=np.float64)
    )
    dn = (
        np.asarray(models["dn"].predict(x), dtype=np.float64)
        if models.get("dn") is not None
        else np.zeros(n, dtype=np.float64)
    )
    return p_long, p_short, vol.reshape(-1), up.reshape(-1), dn.reshape(-1)


def _build_policy_mlp(n_in: int, hidden: Tuple[int, int]) -> Any:
    import torch.nn as nn

    class PolicyMlp(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            h1, h2 = int(hidden[0]), int(hidden[1])
            self.backbone = nn.Sequential(
                nn.Linear(int(n_in), h1),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(h1, h2),
                nn.ReLU(),
            )
            self.opp_long = nn.Linear(h2, 3)
            self.opp_short = nn.Linear(h2, 3)
            self.vol = nn.Linear(h2, 1)
            self.exc = nn.Linear(h2, 2)

        def forward(self, x: Any) -> Tuple[Any, Any, Any, Any]:
            h = self.backbone(x)
            return (
                self.opp_long(h),
                self.opp_short(h),
                self.vol(h).squeeze(-1),
                self.exc(h),
            )

    return PolicyMlp()


def _fit_policy_mlp(
    x: np.ndarray,
    y_opp: np.ndarray,
    y_opp_short: np.ndarray,
    y_vol: np.ndarray,
    y_up: np.ndarray,
    y_dn: np.ndarray,
    *,
    seed: int,
    epochs: int = MLP_EPOCHS,
) -> Optional[Dict[str, Any]]:
    if not torch_available() or x.shape[0] < 32:
        return None
    import torch
    import torch.nn as nn

    torch.manual_seed(int(seed))
    vol_mean = float(np.mean(y_vol))
    vol_std = float(np.std(y_vol))
    vol_std = vol_std if vol_std > 1e-8 else 1.0
    exc = np.column_stack([y_up, y_dn]).astype(np.float32)
    exc_mean = exc.mean(axis=0, keepdims=True)
    exc_std = exc.std(axis=0, keepdims=True)
    exc_std = np.where(exc_std < 1e-8, 1.0, exc_std)
    model = _build_policy_mlp(x.shape[1], MLP_HIDDEN)
    opt = torch.optim.Adam(model.parameters(), lr=MLP_LR)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()
    xt = torch.from_numpy(np.asarray(x, dtype=np.float32))
    yo = torch.from_numpy(np.asarray(y_opp, dtype=np.int64))
    ys = torch.from_numpy(np.asarray(y_opp_short, dtype=np.int64))
    yv = torch.from_numpy(((y_vol - vol_mean) / vol_std).astype(np.float32))
    ye = torch.from_numpy(((exc - exc_mean) / exc_std).astype(np.float32))
    n = xt.shape[0]
    batch = min(int(MLP_BATCH), n)
    model.train()
    for _epoch in range(int(epochs)):
        perm = torch.randperm(n)
        for start in range(0, n, batch):
            idx = perm[start : start + batch]
            opt.zero_grad()
            logits_l, logits_s, vhat, ehat = model(xt[idx])
            loss = (
                ce(logits_l, yo[idx])
                + ce(logits_s, ys[idx])
                + mse(vhat, yv[idx])
                + mse(ehat, ye[idx])
            )
            loss.backward()
            opt.step()
    model.eval()
    return {
        "kind": "mlp",
        "model": model,
        "vol_mean": vol_mean,
        "vol_std": vol_std,
        "exc_mean": exc_mean.reshape(-1),
        "exc_std": exc_std.reshape(-1),
    }


def _predict_mlp_heads(
    models: Mapping[str, Any], x: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    import torch
    import torch.nn.functional as F

    n = x.shape[0]
    if models is None or models.get("model") is None:
        p = np.full((n, 3), 1.0 / 3.0)
        z = np.zeros(n, dtype=np.float64)
        return p, p.copy(), z, z, z
    model = models["model"]
    model.eval()
    with torch.no_grad():
        xt = torch.from_numpy(np.asarray(x, dtype=np.float32))
        logits_l, logits_s, vhat, ehat = model(xt)
        p_long = F.softmax(logits_l, dim=1).cpu().numpy().astype(np.float64)
        p_short = F.softmax(logits_s, dim=1).cpu().numpy().astype(np.float64)
        vol = (
            vhat.cpu().numpy().reshape(-1) * float(models["vol_std"])
            + float(models["vol_mean"])
        )
        exc = ehat.cpu().numpy().astype(np.float64)
        up = exc[:, 0] * float(models["exc_std"][0]) + float(models["exc_mean"][0])
        dn = exc[:, 1] * float(models["exc_std"][1]) + float(models["exc_mean"][1])
    return p_long, p_short, vol.reshape(-1), up.reshape(-1), dn.reshape(-1)


def _predict_heads(
    models: Mapping[str, Any], x: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if str(models.get("kind")) == "mlp":
        return _predict_mlp_heads(models, x)
    return _predict_xgb_heads(models, x)


def _arrays_for_policy(frame: pd.DataFrame) -> Dict[str, np.ndarray]:
    return {
        "x": frame.loc[:, list(FIXED_FEATURE_COLS)].to_numpy(dtype=np.float64),
        "y_opp": frame["y_opportunity"].to_numpy(dtype=np.int64),
        "y_opp_s": frame["y_opportunity_short"].to_numpy(dtype=np.int64),
        "y_vol": frame["y_volatility"].to_numpy(dtype=np.float64),
        "y_up": frame["y_upside"].to_numpy(dtype=np.float64),
        "y_dn": frame["y_downside"].to_numpy(dtype=np.float64),
        "tp": frame["tp_pct"].to_numpy(dtype=np.float64),
        "sl": frame["sl_pct"].to_numpy(dtype=np.float64),
        "fwd": frame["fwd_return_pct"].to_numpy(dtype=np.float64),
    }


def _unfiltered_sides(
    p_long: np.ndarray, p_short: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    long_t = np.argmax(p_long, axis=1) == OPP_TP_FIRST
    short_t = np.argmax(p_short, axis=1) == OPP_TP_FIRST
    conflict = long_t & short_t
    return long_t & ~conflict, short_t & ~conflict


def _fit_kind_models(
    kind: str,
    x_fit: np.ndarray,
    y_long: np.ndarray,
    y_short: np.ndarray,
    y_vol: np.ndarray,
    y_up: np.ndarray,
    y_dn: np.ndarray,
    *,
    config: Mapping[str, Any],
    n_estimators: int,
    seed: int,
) -> Optional[Dict[str, Any]]:
    if kind == "mlp":
        return _fit_policy_mlp(
            x_fit, y_long, y_short, y_vol, y_up, y_dn, seed=seed
        )
    return _fit_xgb_policy_heads(
        x_fit,
        y_long,
        y_short,
        y_vol,
        y_up,
        y_dn,
        config=config,
        n_estimators=n_estimators,
        seed=seed,
    )


def _pooled_side_stats(
    rows: Sequence[Mapping[str, Any]], r_key: str, n_key: str
) -> Tuple[float, float]:
    n_sum = float(sum(float(r.get(n_key, 0.0) or 0.0) for r in rows))
    rs = [
        float(r[r_key])
        for r in rows
        if r_key in r
        and np.isfinite(r[r_key])
        and r[r_key] > float("-inf") / 2
    ]
    mean_r = float(np.mean(rs)) if rs else float("-inf")
    return mean_r, n_sum


def _walk_policy(
    frame: pd.DataFrame,
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
    model_kind: str,
    y_opp_override: Optional[np.ndarray] = None,
    y_opp_short_override: Optional[np.ndarray] = None,
    inner_calibration_split: Optional[
        Callable[..., Tuple[np.ndarray, np.ndarray]]
    ] = None,
) -> Dict[str, Any]:
    usable = _usable_policy(frame)
    pack = _arrays_for_policy(frame)
    if y_opp_override is not None:
        pack["y_opp"] = np.asarray(y_opp_override, dtype=np.int64)
    if y_opp_short_override is not None:
        pack["y_opp_s"] = np.asarray(y_opp_short_override, dtype=np.int64)
    n = len(frame)
    cost = float(config["round_trip_cost_pct"])
    min_n = int(config.get("min_signals", MIN_SIGNALS_DEFAULT))
    calib_frac = float(config.get("calib_fraction", 0.15))
    purge = int(config.get("purge_bars", FROZEN_HORIZON_BARS + 1))
    splitter = inner_calibration_split or inner_split_train
    fold_rows: List[Dict[str, Any]] = []
    unf_rows: List[Dict[str, Any]] = []
    always_long_rows: List[Dict[str, Any]] = []
    always_short_rows: List[Dict[str, Any]] = []
    last_models: Dict[str, Any] = {}
    last_thr: Dict[str, float] = {}
    last_mean: Optional[np.ndarray] = None
    last_std: Optional[np.ndarray] = None
    y_true_l = frame["y_opportunity"].to_numpy(dtype=np.int64)
    y_true_s = frame["y_opportunity_short"].to_numpy(dtype=np.int64)
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_idx, calib_idx = splitter(
            np.asarray(tr_idx, dtype=np.int64),
            calib_fraction=calib_frac,
            purge_bars=purge,
        )
        fit_m = _in_split(n, fit_idx, usable)
        calib_m = _in_split(n, calib_idx, usable)
        eval_m = _in_split(n, va_idx, usable)
        if int(fit_m.sum()) < 32 or int(eval_m.sum()) < 8 or not calib_m.any():
            continue
        y_fit_l = pack["y_opp"][fit_m]
        y_fit_s = pack["y_opp_s"][fit_m]
        if len(np.unique(y_fit_l[y_fit_l >= 0])) < 2:
            continue
        if len(np.unique(y_fit_s[y_fit_s >= 0])) < 2:
            continue
        mean, std = _fit_scaler(pack["x"][fit_m])
        x_fit = _scale_x(pack["x"][fit_m], mean, std)
        models = _fit_kind_models(
            model_kind,
            x_fit,
            y_fit_l,
            y_fit_s,
            pack["y_vol"][fit_m],
            pack["y_up"][fit_m],
            pack["y_dn"][fit_m],
            config=config,
            n_estimators=n_estimators,
            seed=int(seed) + fold_i,
        )
        if models is None:
            continue
        p_lc, p_sc, v_c, u_c, d_c = _predict_heads(
            models, _scale_x(pack["x"][calib_m], mean, std)
        )
        p_le, p_se, v_e, u_e, d_e = _predict_heads(
            models, _scale_x(pack["x"][eval_m], mean, std)
        )
        pnl_lc = long_path_pnl(
            y_true_l[calib_m],
            pack["tp"][calib_m],
            pack["sl"][calib_m],
            pack["fwd"][calib_m],
            cost=cost,
        )
        pnl_sc = short_path_pnl(
            y_true_s[calib_m],
            pack["tp"][calib_m],
            pack["sl"][calib_m],
            pack["fwd"][calib_m],
            cost=cost,
        )
        pnl_le = long_path_pnl(
            y_true_l[eval_m],
            pack["tp"][eval_m],
            pack["sl"][eval_m],
            pack["fwd"][eval_m],
            cost=cost,
        )
        pnl_se = short_path_pnl(
            y_true_s[eval_m],
            pack["tp"][eval_m],
            pack["sl"][eval_m],
            pack["fwd"][eval_m],
            cost=cost,
        )
        thr = choose_twosided_thresholds(
            p_lc[:, OPP_TP_FIRST],
            p_sc[:, OPP_TP_FIRST],
            v_c,
            u_c,
            d_c,
            pack["tp"][calib_m],
            pack["sl"][calib_m],
            pnl_lc,
            pnl_sc,
            min_signals=min_n,
        )
        long_t, short_t = apply_twosided_policy(
            p_le[:, OPP_TP_FIRST],
            p_se[:, OPP_TP_FIRST],
            v_e,
            u_e,
            d_e,
            pack["tp"][eval_m],
            pack["sl"][eval_m],
            thr_long=float(thr["thr_long"]),
            thr_short=float(thr["thr_short"]),
            k_vol=float(thr["k_vol"]),
            k_dd=float(thr["k_dd"]),
        )
        scored = score_twosided_r(
            long_t, short_t, pnl_le, pnl_se, pack["sl"][eval_m], min_n
        )
        unf_l, unf_s = _unfiltered_sides(p_le, p_se)
        unf = score_twosided_r(
            unf_l, unf_s, pnl_le, pnl_se, pack["sl"][eval_m], min_n
        )
        always_l = score_policy_r(
            np.ones(int(eval_m.sum()), dtype=bool),
            pnl_le,
            pack["sl"][eval_m],
            min_n,
        )
        always_s = score_policy_r(
            np.ones(int(eval_m.sum()), dtype=bool),
            pnl_se,
            pack["sl"][eval_m],
            min_n,
        )
        row = {
            "fold": fold_i,
            "kind": model_kind,
            "thr_long": thr["thr_long"],
            "thr_short": thr["thr_short"],
            "k_vol": thr["k_vol"],
            "k_dd": thr["k_dd"],
            **scored,
        }
        fold_rows.append(row)
        unf_rows.append({"fold": fold_i, **unf})
        always_long_rows.append({"fold": fold_i, **always_l})
        always_short_rows.append({"fold": fold_i, **always_s})
        last_models = dict(models)
        last_thr = dict(thr)
        last_mean, last_std = mean, std
        print(
            f"{model_kind} policy fold {fold_i}: R={scored['r_expectancy']} "
            f"n={scored['n']:.0f} long_n={scored['long_n']:.0f} "
            f"short_n={scored['short_n']:.0f} unfiltered={unf['r_expectancy']}"
        )
        if model_kind == "mlp":
            ll = _log_loss(y_true_l[eval_m], p_le, n_classes=3)
            ls = _log_loss(y_true_s[eval_m], p_se, n_classes=3)
            prior_l = _class_prior(y_fit_l, 3)
            prior_s = _class_prior(y_fit_s, 3)
            n_e = int(eval_m.sum())
            base_l = _log_loss(
                y_true_l[eval_m],
                np.repeat(prior_l[None, :], n_e, axis=0),
                n_classes=3,
            )
            base_s = _log_loss(
                y_true_s[eval_m],
                np.repeat(prior_s[None, :], n_e, axis=0),
                n_classes=3,
            )
            row["opp_long_log_loss"] = ll
            row["opp_long_baseline_log_loss"] = base_l
            row["opp_short_log_loss"] = ls
            row["opp_short_baseline_log_loss"] = base_s
            row["vol_rmse"] = _rmse(pack["y_vol"][eval_m], v_e)
            row["down_rmse"] = _rmse(pack["y_dn"][eval_m], d_e)
            row["up_rmse"] = _rmse(pack["y_up"][eval_m], u_e)
    long_r, long_n = _pooled_side_stats(
        fold_rows, "long_r_expectancy", "long_n"
    )
    short_r, short_n = _pooled_side_stats(
        fold_rows, "short_r_expectancy", "short_n"
    )
    return {
        "kind": model_kind,
        "wf_r_expectancy": _mean_finite_r(fold_rows),
        "wf_positive_share": (
            float(
                np.mean(
                    [
                        float(r["r_expectancy"]) > 0.0
                        for r in fold_rows
                        if np.isfinite(r["r_expectancy"])
                        and r["r_expectancy"] > float("-inf") / 2
                    ]
                    or [np.nan]
                )
            )
        ),
        "n_folds_scored": sum(
            1
            for r in fold_rows
            if np.isfinite(r["r_expectancy"])
            and r["r_expectancy"] > float("-inf") / 2
        ),
        "unfiltered_r": _mean_finite_r(unf_rows),
        "always_long_r": _mean_finite_r(always_long_rows),
        "always_short_r": _mean_finite_r(always_short_rows),
        "long_r_expectancy": long_r,
        "short_r_expectancy": short_r,
        "long_n": long_n,
        "short_n": short_n,
        "fold_rows": fold_rows,
        "unfiltered_folds": unf_rows,
        "always_long_folds": always_long_rows,
        "always_short_folds": always_short_rows,
        "models": last_models,
        "thr": last_thr,
        "mean": last_mean,
        "std": last_std,
    }


def _confirm_policy_test(
    frame: pd.DataFrame,
    splits: Mapping[str, np.ndarray],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seed: int,
    model_kind: str,
    inner_calibration_split: Optional[
        Callable[..., Tuple[np.ndarray, np.ndarray]]
    ] = None,
) -> Dict[str, Any]:
    usable = _usable_policy(frame)
    pack = _arrays_for_policy(frame)
    n = len(frame)
    cost = float(config["round_trip_cost_pct"])
    min_n = int(config.get("min_signals", MIN_SIGNALS_DEFAULT))
    splitter = inner_calibration_split or inner_split_train
    fit_idx, calib_idx = splitter(
        np.asarray(splits["train"], dtype=np.int64),
        calib_fraction=float(config.get("calib_fraction", 0.15)),
        purge_bars=int(config.get("purge_bars", FROZEN_HORIZON_BARS + 1)),
    )
    val_idx = np.asarray(splits["val"], dtype=np.int64)
    if val_idx.size:
        calib_idx = val_idx
    fit_m = _in_split(n, fit_idx, usable)
    calib_m = _in_split(n, calib_idx, usable)
    test_m = _in_split(n, splits["test"], usable)
    if int(fit_m.sum()) < 32 or int(test_m.sum()) < 8 or not calib_m.any():
        return {"kind": model_kind, "test_scored": False}
    mean, std = _fit_scaler(pack["x"][fit_m])
    x_fit = _scale_x(pack["x"][fit_m], mean, std)
    models = _fit_kind_models(
        model_kind,
        x_fit,
        pack["y_opp"][fit_m],
        pack["y_opp_s"][fit_m],
        pack["y_vol"][fit_m],
        pack["y_up"][fit_m],
        pack["y_dn"][fit_m],
        config=config,
        n_estimators=n_estimators,
        seed=seed,
    )
    if models is None:
        return {"kind": model_kind, "test_scored": False}
    p_lc, p_sc, v_c, u_c, d_c = _predict_heads(
        models, _scale_x(pack["x"][calib_m], mean, std)
    )
    p_lt, p_st, v_t, u_t, d_t = _predict_heads(
        models, _scale_x(pack["x"][test_m], mean, std)
    )
    pnl_lc = long_path_pnl(
        pack["y_opp"][calib_m],
        pack["tp"][calib_m],
        pack["sl"][calib_m],
        pack["fwd"][calib_m],
        cost=cost,
    )
    pnl_sc = short_path_pnl(
        pack["y_opp_s"][calib_m],
        pack["tp"][calib_m],
        pack["sl"][calib_m],
        pack["fwd"][calib_m],
        cost=cost,
    )
    pnl_lt = long_path_pnl(
        pack["y_opp"][test_m],
        pack["tp"][test_m],
        pack["sl"][test_m],
        pack["fwd"][test_m],
        cost=cost,
    )
    pnl_st = short_path_pnl(
        pack["y_opp_s"][test_m],
        pack["tp"][test_m],
        pack["sl"][test_m],
        pack["fwd"][test_m],
        cost=cost,
    )
    thr = choose_twosided_thresholds(
        p_lc[:, OPP_TP_FIRST],
        p_sc[:, OPP_TP_FIRST],
        v_c,
        u_c,
        d_c,
        pack["tp"][calib_m],
        pack["sl"][calib_m],
        pnl_lc,
        pnl_sc,
        min_signals=min_n,
    )
    long_t, short_t = apply_twosided_policy(
        p_lt[:, OPP_TP_FIRST],
        p_st[:, OPP_TP_FIRST],
        v_t,
        u_t,
        d_t,
        pack["tp"][test_m],
        pack["sl"][test_m],
        thr_long=float(thr["thr_long"]),
        thr_short=float(thr["thr_short"]),
        k_vol=float(thr["k_vol"]),
        k_dd=float(thr["k_dd"]),
    )
    scored = score_twosided_r(
        long_t, short_t, pnl_lt, pnl_st, pack["sl"][test_m], min_n
    )
    return {
        "kind": model_kind,
        "test_scored": True,
        "n_test": float(test_m.sum()),
        "thr_long": thr["thr_long"],
        "thr_short": thr["thr_short"],
        "k_vol": thr["k_vol"],
        "k_dd": thr["k_dd"],
        **{f"test_{k}": v for k, v in scored.items()},
    }


def run_policy_and_multihead(
    bars: pd.DataFrame,
    *,
    config: Mapping[str, Any],
    splits: Mapping[str, np.ndarray],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    export_dir: Path,
    inner_calibration_split: Optional[
        Callable[..., Tuple[np.ndarray, np.ndarray]]
    ] = None,
    n_estimators: Optional[int] = None,
) -> Dict[str, Any]:
    """XGB long/short policy, then a shared MLP, on the frozen 50-feature set.

    Direction and return heads are not fit. Live fusion is not written.
    """
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except Exception:
        pass
    trees = int(n_estimators or config.get("n_estimators", N_ESTIMATORS_DEFAULT))
    seed = int(config.get("seed", SCREEN_SEED))
    rng = np.random.default_rng(seed)
    featured = add_fixed_ohlcv_features(
        bars, atr_period=int(config.get("atr_period", ATR_PERIOD_DEFAULT))
    )
    labeled = attach_fixed_targets(
        featured,
        horizon_bars=int(config.get("horizon_bars", FROZEN_HORIZON_BARS)),
        tp_atr_mult=TP_ATR_MULT,
        sl_atr_mult=SL_ATR_MULT,
        dir_band_atr_mult=DIR_BAND_ATR_MULT,
        atr_period=int(config.get("atr_period", ATR_PERIOD_DEFAULT)),
    )
    print("n_features", len(FIXED_FEATURE_COLS))
    print("policy heads", list(POLICY_HEAD_IDS))
    print("opportunity_side", "long_and_short")
    double_n = int(np.nansum(labeled["same_bar_double_touch"].to_numpy()))
    double_s = int(np.nansum(labeled["same_bar_double_touch_short"].to_numpy()))

    print("\n===== XGB policy =====")
    xgb_wf = _walk_policy(
        labeled,
        wf_folds,
        config,
        n_estimators=trees,
        seed=seed,
        model_kind="xgb",
        inner_calibration_split=inner_calibration_split,
    )
    usable = _usable_policy(labeled)
    block = int(config.get("horizon_bars", FROZEN_HORIZON_BARS))
    shuffled_l = shuffle_labeled_target(
        labeled["y_opportunity"].to_numpy(dtype=np.int64),
        labeled=usable,
        block=block,
        rng=rng,
    )
    shuffled_s = shuffle_labeled_target(
        labeled["y_opportunity_short"].to_numpy(dtype=np.int64),
        labeled=usable,
        block=block,
        rng=rng,
    )
    print("\n===== XGB shuffle null =====")
    xgb_shuf = _walk_policy(
        labeled,
        wf_folds,
        config,
        n_estimators=trees,
        seed=seed + 100,
        model_kind="xgb",
        y_opp_override=shuffled_l,
        y_opp_short_override=shuffled_s,
        inner_calibration_split=inner_calibration_split,
    )
    min_n = int(config.get("min_signals", MIN_SIGNALS_DEFAULT))
    xgb_gate = policy_test_gate(
        [r["r_expectancy"] for r in xgb_wf["fold_rows"]],
        float(xgb_wf["unfiltered_r"]),
        float(xgb_shuf["wf_r_expectancy"]),
        long_r=float(xgb_wf["long_r_expectancy"]),
        short_r=float(xgb_wf["short_r_expectancy"]),
        long_n=float(xgb_wf["long_n"]),
        short_n=float(xgb_wf["short_n"]),
        min_signals=min_n,
    )
    test_rows: List[Dict[str, Any]] = []
    if xgb_gate:
        print("Test once XGB policy")
        test_rows.append(
            _confirm_policy_test(
                labeled,
                splits,
                config,
                n_estimators=trees,
                seed=seed,
                model_kind="xgb",
                inner_calibration_split=inner_calibration_split,
            )
        )

    mlp_wf: Dict[str, Any] = {"kind": "mlp", "skipped": True}
    mlp_shuf: Dict[str, Any] = {"kind": "mlp", "skipped": True}
    if torch_available():
        print("\n===== MLP policy =====")
        mlp_wf = _walk_policy(
            labeled,
            wf_folds,
            config,
            n_estimators=trees,
            seed=seed,
            model_kind="mlp",
            inner_calibration_split=inner_calibration_split,
        )
        print("\n===== MLP shuffle null =====")
        mlp_shuf = _walk_policy(
            labeled,
            wf_folds,
            config,
            n_estimators=trees,
            seed=seed + 100,
            model_kind="mlp",
            y_opp_override=shuffled_l,
            y_opp_short_override=shuffled_s,
            inner_calibration_split=inner_calibration_split,
        )
        mlp_gate = policy_test_gate(
            [r["r_expectancy"] for r in mlp_wf.get("fold_rows") or []],
            float(mlp_wf.get("unfiltered_r", float("-inf"))),
            float(mlp_shuf.get("wf_r_expectancy", float("-inf"))),
            long_r=float(mlp_wf.get("long_r_expectancy", float("-inf"))),
            short_r=float(mlp_wf.get("short_r_expectancy", float("-inf"))),
            long_n=float(mlp_wf.get("long_n", 0.0)),
            short_n=float(mlp_wf.get("short_n", 0.0)),
            min_signals=min_n,
        )
        if mlp_gate:
            print("Test once MLP policy")
            test_rows.append(
                _confirm_policy_test(
                    labeled,
                    splits,
                    config,
                    n_estimators=trees,
                    seed=seed,
                    model_kind="mlp",
                    inner_calibration_split=inner_calibration_split,
                )
            )
    else:
        print("Skipping Stage B MLP: torch is not available")
        mlp_gate = False

    compare = [
        {
            "model": "xgb_policy",
            "wf_r_expectancy": xgb_wf["wf_r_expectancy"],
            "wf_positive_share": xgb_wf["wf_positive_share"],
            "long_r": xgb_wf["long_r_expectancy"],
            "short_r": xgb_wf["short_r_expectancy"],
            "long_n": xgb_wf["long_n"],
            "short_n": xgb_wf["short_n"],
            "unfiltered_r": xgb_wf["unfiltered_r"],
            "always_long_r": xgb_wf["always_long_r"],
            "always_short_r": xgb_wf["always_short_r"],
            "shuffle_r": xgb_shuf["wf_r_expectancy"],
            "test_gate": xgb_gate,
        },
        {
            "model": "mlp_policy",
            "wf_r_expectancy": mlp_wf.get("wf_r_expectancy"),
            "wf_positive_share": mlp_wf.get("wf_positive_share"),
            "long_r": mlp_wf.get("long_r_expectancy"),
            "short_r": mlp_wf.get("short_r_expectancy"),
            "long_n": mlp_wf.get("long_n"),
            "short_n": mlp_wf.get("short_n"),
            "unfiltered_r": mlp_wf.get("unfiltered_r"),
            "always_long_r": mlp_wf.get("always_long_r"),
            "always_short_r": mlp_wf.get("always_short_r"),
            "shuffle_r": mlp_shuf.get("wf_r_expectancy"),
            "test_gate": mlp_gate,
            "skipped": mlp_wf.get("skipped", False),
        },
    ]
    _print_stage_frame("XGB policy folds", xgb_wf.get("fold_rows") or [])
    _print_stage_frame("XGB unfiltered", xgb_wf.get("unfiltered_folds") or [])
    _print_stage_frame("XGB shuffle", xgb_shuf.get("fold_rows") or [])
    _print_stage_frame("MLP policy folds", mlp_wf.get("fold_rows") or [])
    _print_stage_frame("Compare", compare)
    _print_stage_frame("Test once", test_rows)
    print("xgb_test_gate", xgb_gate, "mlp_test_gate", mlp_gate)
    print("opportunity_double_touch_n", double_n)
    print("opportunity_short_double_touch_n", double_s)

    payload = {
        "resolution": str(config.get("resolution", FROZEN_RESOLUTION)),
        "horizon_bars": int(config.get("horizon_bars", FROZEN_HORIZON_BARS)),
        "n_features": len(FIXED_FEATURE_COLS),
        "feature_cols": list(FIXED_FEATURE_COLS),
        "policy_heads": list(POLICY_HEAD_IDS),
        "tp_atr_mult": TP_ATR_MULT,
        "sl_atr_mult": SL_ATR_MULT,
        "entry": "close[t]",
        "opportunity_side": "long_and_short",
        "opportunity_double_touch_n": double_n,
        "opportunity_short_double_touch_n": double_s,
        "xgb_policy": {k: v for k, v in xgb_wf.items() if k != "models"},
        "xgb_shuffle": {k: v for k, v in xgb_shuf.items() if k != "models"},
        "mlp_policy": {k: v for k, v in mlp_wf.items() if k != "models"},
        "mlp_shuffle": {k: v for k, v in mlp_shuf.items() if k != "models"},
        "compare": compare,
        "test_rows": test_rows,
        "xgb_test_gate": xgb_gate,
        "mlp_test_gate": mlp_gate,
    }
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)
    (export_dir / "policy.json").write_text(
        json.dumps(_json_sanitize(payload), indent=2),
        encoding="utf-8",
    )
    print("wrote", export_dir / "policy.json")
    return payload


In [ ]:
# CELL: FL_RUN
from pathlib import Path
import pandas as pd

_sweep_dir = Path(str(CONFIG["export_dir"]))
_sweep_dir.mkdir(parents=True, exist_ok=True)
sweep_payload = run_policy_and_multihead(
    bars,
    config=CONFIG,
    splits=splits,
    wf_folds=wf_folds,
    export_dir=_sweep_dir,
    inner_calibration_split=inner_calibration_split,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


def _show_stage(name: str, rows: object) -> None:
    print(f"\n===== {name} =====")
    frame = pd.DataFrame(list(rows or []))
    if frame.empty:
        print("(empty)")
        return
    nested = [
        c
        for c in frame.columns
        if frame[c].map(lambda v: isinstance(v, (dict, list))).any()
    ]
    show = frame.drop(columns=nested) if nested else frame
    print(show.to_string(index=False))


print("n_features", sweep_payload.get("n_features"))
print("policy_heads", sweep_payload.get("policy_heads"))
_show_stage("XGB policy folds", (sweep_payload.get("xgb_policy") or {}).get("fold_rows"))
_show_stage(
    "XGB unfiltered",
    (sweep_payload.get("xgb_policy") or {}).get("unfiltered_folds"),
)
_show_stage("XGB shuffle", (sweep_payload.get("xgb_shuffle") or {}).get("fold_rows"))
_show_stage("MLP policy folds", (sweep_payload.get("mlp_policy") or {}).get("fold_rows"))
_show_stage("Compare", sweep_payload.get("compare"))
_show_stage("Test once", sweep_payload.get("test_rows"))
print("xgb_test_gate", sweep_payload.get("xgb_test_gate"))
print("mlp_test_gate", sweep_payload.get("mlp_test_gate"))
print("opportunity_double_touch_n", sweep_payload.get("opportunity_double_touch_n"))
import shutil

_zip_path = shutil.make_archive(str(_sweep_dir), "zip", root_dir=str(_sweep_dir))
print("rewrote", _zip_path)
try:
    from google.colab import files

    files.download(str(_sweep_dir) + ".zip")
except ImportError:
    print("Not Colab; files remain under", _sweep_dir)


n_features 50
policy heads ['opportunity_head', 'opportunity_short_head', 'volatility_head', 'excursion_head']
opportunity_side long_and_short

===== XGB policy =====


xgb policy fold 1: R=-0.20073453648482484 n=706 long_n=106 short_n=600 unfiltered=-0.08413982184732108


xgb policy fold 2: R=-0.37485383493769486 n=389 long_n=96 short_n=293 unfiltered=-0.4274971977240109


xgb policy fold 3: R=-0.18836952201603555 n=211 long_n=20 short_n=191 unfiltered=0.024248233380642498

===== XGB shuffle null =====


xgb policy fold 1: R=-0.22401061572327471 n=698 long_n=299 short_n=399 unfiltered=-0.27528870942644695


...

mlp policy fold 1: R=-0.32808257455648143 n=516 long_n=31 short_n=485 unfiltered=-0.10163470876934029


mlp policy fold 2: R=-0.18636537564849912 n=436 long_n=396 short_n=40 unfiltered=-inf


mlp policy fold 3: R=-0.25427676762637336 n=284 long_n=204 short_n=80 unfiltered=-0.1751013045755974

===== XGB policy folds =====
 fold kind  thr_long  thr_short  k_vol  k_dd  r_expectancy     tstat     n  long_r_expectancy  long_n  short_r_expectancy  short_n
    1  xgb  0.307043   0.193893   0.15   1.2     -0.200735 -4.408603 706.0          -0.298248   106.0           -0.183507    600.0
    2  xgb  0.169035   0.207039   0.15   1.0     -0.374854 -6.898094 389.0               -inf    96.0           -0.418322    293.0
    3  xgb  0.402998   0.300791   0.15   1.2     -0.188370 -2.048307 211.0               -inf    20.0           -0.199213    191.0

===== XGB unfiltered =====
 fold  r_expectancy      tstat      n  long_r_expectancy  long_n  short_r_expectancy  short_n
    1     -0.084140  -1.936339  925.0          -0.248376   316.0            0.001080    609.0
    2     -0.427497 -12.419444 1311.0          -0.462482   455.0           -0.408901    856.0
    3      0.024248   0.329862  354

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*File Operation*: `download` on `/content/export/opportunity_30m_h24_twosided.zip`